# Polyglot: speak your language, hear theirs

**Veritas AI International High School Hackathon 2026 · Track: AI + Languages**

Polyglot turns speech in one language into speech in another, so two people who share no language can talk. This notebook is the AI behind it, and the test of how well it really works.

**Our question.** People in India mix Hindi and English inside one sentence: *"Kal meeting hai, so please complete the presentation."* Does speech translation still work on that, and can we make it work better?

| Part | What it does |
|---|---|
| 1 to 5 | Builds the pipeline from free, open models: Whisper (speech to text), NLLB-200 (translation), MMS-TTS (voice) |
| 6 to 10 | Tests it on the same sentences spoken in Hindi, Tamil, Bengali and English, against simple baselines |
| 11 to 14 | Tests it on our own Hinglish recordings and picks the best way to handle them |
| 15 to 17 | Shows the worst mistakes side by side, saves the results, and runs a live demo |

**To run it:** `Runtime` → `Change runtime type` → `T4 GPU`, then `Runtime` → `Run all`. A full run took about 25 minutes with the small speech model and takes longer with the big one. Tick `QUICK_TEST` in the first cell for a short run on 10 sentences per language.

In [ ]:
#@title 1 · Setup and settings
import os, sys, subprocess, importlib, importlib.util, warnings, json, time, re, random, math, unicodedata, glob, tempfile
warnings.filterwarnings("ignore")
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

def ensure(*pkgs):
    """Install anything that is missing."""
    missing = [p for p in pkgs if importlib.util.find_spec(p) is None]
    if missing:
        print("Installing", ", ".join(missing), "...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()

ensure("sacrebleu", "jiwer", "rapidfuzz", "soundfile", "gradio")

# ---------------- settings you can change ----------------
QUICK_TEST = False        #@param {type:"boolean"}
N_PER_LANG = 100          #@param {type:"integer"}
USE_GOOGLE_DRIVE = True   #@param {type:"boolean"}
ASR_MODEL = "openai/whisper-large-v3"  #@param ["openai/whisper-small", "openai/whisper-medium", "openai/whisper-large-v3"]
COMPARE_ASR_MODEL = "openai/whisper-small"  #@param ["", "openai/whisper-small", "openai/whisper-medium", "openai/whisper-large-v3", "openai/whisper-large-v3-turbo"]
HINGLISH_SHEET = ""       #@param {type:"string"}
RUN_RECORDER = False      #@param {type:"boolean"}
RUN_RATER = True          #@param {type:"boolean"}
RUN_DEMO = True           #@param {type:"boolean"}
ELEVEN_VOICES = 2         #@param {type:"integer"}
ELEVEN_VOICE_NAMES = ""   #@param {type:"string"}
ELEVEN_MODEL = "eleven_v4"  #@param ["eleven_v4", "eleven_v3", "eleven_multilingual_v2"]
# ----------------------------------------------------------
LANGS_TESTED = ["hi", "ta", "bn", "en"]
SEED = 13
if QUICK_TEST:
    N_PER_LANG = 10
N_PER_LANG = int(os.environ.get("POLYGLOT_N", N_PER_LANG))

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DATA_DIR = "/content/polyglot" if IN_COLAB else os.path.abspath("polyglot_data")
if IN_COLAB and USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DATA_DIR = "/content/drive/MyDrive/polyglot"
    except Exception as e:
        print("Google Drive was not connected, so files are kept only until this session ends.", e)
DATA_DIR = os.environ.get("POLYGLOT_DATA", DATA_DIR)
RESULTS_DIR = os.path.join(DATA_DIR, "results")
HINGLISH_DIR = os.path.join(DATA_DIR, "hinglish")
CONFIRM_DIR = os.path.join(DATA_DIR, "hinglish_confirm")     # new sentences for the second Hinglish test
for d in (RESULTS_DIR, os.path.join(HINGLISH_DIR, "audio"), os.path.join(CONFIRM_DIR, "audio")):
    os.makedirs(d, exist_ok=True)

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
if IN_COLAB and not GEMINI_API_KEY:
    try:
        from google.colab import userdata
        GEMINI_API_KEY = userdata.get("GEMINI_API_KEY") or ""
    except Exception:
        pass

ELEVENLABS_API_KEY = os.environ.get("ELEVENLABS_API_KEY", "")
if IN_COLAB and not ELEVENLABS_API_KEY:
    try:
        from google.colab import userdata
        ELEVENLABS_API_KEY = userdata.get("ELEVENLABS_API_KEY") or ""
    except Exception:
        pass

print("Sentences per language:", N_PER_LANG)
print("Data folder:", DATA_DIR)
print("Gemini comparison:", "on" if GEMINI_API_KEY else "off (add a Colab secret called GEMINI_API_KEY to turn it on)")
print("Computer-made Hinglish voices:", "on" if ELEVENLABS_API_KEY else "off (add a Colab secret called ELEVENLABS_API_KEY to turn it on)")

## 1b · Hinglish sentences, and computer-made voices for them

Our Hinglish sentences travel inside this notebook. If an ElevenLabs key is set, this cell asks ElevenLabs (a text-to-speech service) to read every sentence aloud and saves the clips next to any recordings made by people. Those clips are **computer-made speech, not real speakers**. Their file names start with `tts-`, and every table below keeps them apart from human recordings.

This runs early so that a wrong key or an empty quota shows up in the first minute, not at the end of the run. Clips already on disk are not made again.

In [ ]:
#@title 1b · Load the Hinglish sentences and make computer voices (only if a key is set)
import urllib.request, urllib.error
import pandas as pd
AUDIO_EXT =(".wav", ".m4a", ".mp3", ".ogg", ".webm", ".flac", ".aac", ".opus", ".mp4", ".3gp", ".amr")
DEFAULT_SENTENCES = 'id,type,group,text,english\n1,hinglish,1,"Kal school mein holiday hai kya?","Is tomorrow a holiday at school?"\n3,hinglish,3,"Mummy, mera charger kahan rakha hai?","Mom, where have you kept my charger?"\n5,hinglish,5,"Main five minutes mein niche aa raha hoon.","I am coming downstairs in five minutes."\n8,hinglish,8,"Kal raat ko bahut heavy rain hui.","It rained very heavily last night."\n13,hinglish,13,"Tumne homework complete kar liya?","Have you completed your homework?"\n15,hinglish,15,"Mujhe subah jaldi wake up karna hai.","I have to wake up early in the morning."\n18,hinglish,18,"Please fan off kar do, thand lag rahi hai.","Please turn off the fan, I am feeling cold."\n21,hinglish,21,"Tum kahan ho abhi? Station pe?","Where are you right now? At the station?"\n23,hinglish,23,"Apna wallet aur keys mat bhoolna.","Don\'t forget your wallet and keys."\n27,hinglish,27,"Mujhe headache ho raha hai subah se.","I have been having a headache since morning."\n31,hinglish,31,"Kal meeting hai, so please complete the presentation.","There is a meeting tomorrow, so please complete the presentation."\n33,hinglish,33,"Agar train late hui toh hum cab book kar lenge.","If the train gets delayed, we will book a cab."\n35,hinglish,35,"Teacher ne bola ki assignment Friday tak submit karna hai.","The teacher said that the assignment must be submitted by Friday."\n38,hinglish,38,"Is dress pe 20 percent discount mil raha hai kya?","Is there a 20 percent discount available on this dress?"\n39,hinglish,39,"Network issue ki wajah se meri call drop ho gayi.","My call dropped because of a network issue."\n43,hinglish,43,"Usne mera message seen karke bhi reply nahi kiya.","Even after seeing my message, they didn\'t reply."\n46,hinglish,46,"Kal subah 6 baje ka alarm laga dena, miss nahi hona chahiye.","Set an alarm for 6 AM tomorrow morning, it shouldn\'t be missed."\n48,hinglish,48,"Online order kiya tha, par delivery boy ne abhi tak call nahi kiya.","I had ordered online, but the delivery boy hasn\'t called yet."\n57,hinglish,57,"Garmi itni badh gayi hai ki bina AC ke neend hi nahi aati.","The heat has increased so much that I can\'t even sleep without AC."\n60,hinglish,60,"Kal raat ko light chali gayi thi, isliye phone charge nahi ho paya.","The power went out last night, so my phone couldn\'t get charged."\n61,hinglish,61,"Bhai, scene ye hai ki kal subah 7 baje niklenge warna jam mein phans jayenge.","Bro, the situation is that we\'ll leave at 7 AM tomorrow or else we\'ll get stuck in a traffic jam."\n63,hinglish,63,"Mummy kasam, maine tera charger nahi liya, Rohan se puch le!","I swear on Mom, I didn\'t take your charger, go ask Rohan!"\n65,hinglish,65,"Ek kaam kar, 1,500 rupees Paytm kar de, baaki hisaab baad mein settle karenge.","Do one thing, Paytm me 1,500 rupees, we\'ll settle the rest of the account later."\n66,hinglish,66,"Bhai, bill split kar lete hain, per head 350 rupaye padega.","Bro, let\'s split the bill, it\'ll come to 350 rupees per person."\n69,hinglish,69,"Ananya bol rahi thi ki kal Connaught Place pe milte hain sharp 4:30 baje.","Ananya was saying let\'s meet at Connaught Place tomorrow at 4:30 sharp."\n72,hinglish,72,"Bhai, tu tension mat le, jugad lagake kal tak passes arrange kar dunga.","Bro, don\'t worry, I\'ll find a workaround and arrange passes by tomorrow."\n74,hinglish,74,"Suno, Rajdhani Express platform number 4 pe aa rahi hai, jaldi luggage uthao!","Listen, Rajdhani Express is arriving on platform number 4, quickly pick up the luggage!"\n77,hinglish,77,"Yahan se Airport jaane mein kam se kam dedh ghanta lagega peak hours mein.","Going to the Airport from here will take at least one and a half hours during peak hours."\n78,hinglish,78,"Arre yaar, main toh bilkul bhool hi gaya ki aaj Vikram ka birthday hai!","Oh man, I completely forgot that today is Vikram\'s birthday!"\n79,hinglish,79,"Tune jo link bheja tha woh open hi nahi ho raha, error 404 dikha raha hai.","The link you had sent isn\'t opening at all, it\'s showing error 404."\n101,hindi,1,"कल विद्यालय में छुट्टी है क्या?","Is tomorrow a holiday at school?"\n108,hindi,8,"कल रात को बहुत तेज़ बारिश हुई।","It rained very heavily last night."\n113,hindi,13,"तुमने गृहकार्य पूरा कर लिया?","Have you completed your homework?"\n115,hindi,15,"मुझे सुबह जल्दी उठना है।","I have to wake up early in the morning."\n118,hindi,18,"कृपया पंखा बंद कर दो, ठंड लग रही है।","Please turn off the fan, I am feeling cold."\n123,hindi,23,"अपना बटुआ और चाबियाँ मत भूलना।","Don\'t forget your wallet and keys."\n127,hindi,27,"मुझे सुबह से सिरदर्द हो रहा है।","I have been having a headache since morning."\n138,hindi,38,"क्या इस पोशाक पर बीस प्रतिशत छूट मिल रही है?","Is there a 20 percent discount available on this dress?"\n177,hindi,77,"यहाँ से हवाई अड्डे जाने में व्यस्त समय में कम से कम डेढ़ घंटा लगेगा।","Going to the Airport from here will take at least one and a half hours during peak hours."\n178,hindi,78,"अरे यार, मैं तो बिल्कुल भूल ही गया कि आज विक्रम का जन्मदिन है!","Oh man, I completely forgot that today is Vikram\'s birthday!"\n201,english,1,"Is tomorrow a holiday at school?","Is tomorrow a holiday at school?"\n208,english,8,"It rained very heavily last night.","It rained very heavily last night."\n213,english,13,"Have you completed your homework?","Have you completed your homework?"\n215,english,15,"I have to wake up early in the morning.","I have to wake up early in the morning."\n218,english,18,"Please turn off the fan, I am feeling cold.","Please turn off the fan, I am feeling cold."\n223,english,23,"Don\'t forget your wallet and keys.","Don\'t forget your wallet and keys."\n227,english,27,"I have been having a headache since morning.","I have been having a headache since morning."\n238,english,38,"Is there a 20 percent discount available on this dress?","Is there a 20 percent discount available on this dress?"\n277,english,77,"Going to the Airport from here will take at least one and a half hours during peak hours.","Going to the Airport from here will take at least one and a half hours during peak hours."\n278,english,78,"Oh man, I completely forgot that today is Vikram\'s birthday!","Oh man, I completely forgot that today is Vikram\'s birthday!"\n'
DEFAULT_CONFIRM = 'id,type,group,text,english\n2,hinglish,2,"Mujhe thoda water chahiye, please.","I need some water, please."\n4,hinglish,4,"Aaj lunch mein kya bana hai?","What has been made for lunch today?"\n6,hinglish,6,"Tumhara phone silent pe hai kya?","Is your phone on silent?"\n7,hinglish,7,"Please gate lock kar dena.","Please lock the gate."\n9,hinglish,9,"Yeh bag bahut heavy hai, help karo.","This bag is very heavy, help me."\n10,hinglish,10,"Hum log evening mein market jayenge.","We will go to the market in the evening."\n11,hinglish,11,"Bhai, ek plate momos pack kar do.","Brother, pack one plate of momos."\n12,hinglish,12,"Mera internet connection bahut slow chal raha hai.","My internet connection is running very slow."\n14,hinglish,14,"Aaj traffic bahut zyada hai road pe.","There is too much traffic on the road today."\n16,hinglish,16,"Yeh shirt mujhpe achhi lag rahi hai na?","This shirt looks good on me, doesn\'t it?"\n17,hinglish,17,"Papa office se kab tak aayenge?","By when will Dad return from the office?"\n19,hinglish,19,"Kal humara maths ka test hai.","Tomorrow we have a maths test."\n20,hinglish,20,"Mujhe cold drink nahi peeni hai.","I don\'t want to drink a cold drink."\n22,hinglish,22,"Yeh movie bahut boring thi yaar.","This movie was very boring, buddy."\n24,hinglish,24,"Chai mein sugar thodi kam rakhna.","Keep the sugar a little less in the tea."\n25,hinglish,25,"Main kal library jaunga books return karne.","I will go to the library tomorrow to return books."\n26,hinglish,26,"Door bell baj rahi hai, dekho kaun hai.","The doorbell is ringing, see who it is."\n28,hinglish,28,"Tumhari flight kitne baje land karegi?","At what time will your flight land?"\n29,hinglish,29,"Aaj weather bahut pleasant hai.","The weather is very pleasant today."\n30,hinglish,30,"Please thoda side hona, mujhe nikalna hai.","Please move aside a bit, I need to get through."\n32,hinglish,32,"Auto wale bhaiya ne extra paise charge kiye.","The auto-rickshaw driver charged extra money."\n34,hinglish,34,"Bhai, GPay pe QR code scan nahi ho raha.","Bro, the QR code isn\'t scanning on GPay."\n36,hinglish,36,"Mujhe laga tumne ticket already cancel kar di hogi.","I thought you would have already cancelled the ticket."\n37,hinglish,37,"Mummy ko bol dena ki main dinner bahar karunga.","Tell Mom that I will have dinner outside."\n40,hinglish,40,"Hum Goa trip ka plan next month ke liye postpone kar rahe hain.","We are postponing the Goa trip plan to next month."\n41,hinglish,41,"Aap paneer tikka medium spicy rakhna, zyada mirchi mat dalna.","Keep the paneer tikka medium spicy, don\'t put too much chili."\n42,hinglish,42,"Metro card recharge karwana hai, counter pe bheed dekho!","I need to get my metro card recharged, look at the crowd at the counter!"\n44,hinglish,44,"Exam ke baad sab friends milke movie dekhne jayenge.","After the exam, all friends will get together and go watch a movie."\n45,hinglish,45,"Ye parcel return karna hai kyunki size fit nahi aa raha.","I need to return this parcel because the size doesn\'t fit."\n47,hinglish,47,"Papa ne kaha hai pehle marks dikhao, phir new phone milega.","Dad said show your marks first, then you\'ll get a new phone."\n49,hinglish,49,"Tumhari battery 5 percent pe hai, power bank use kar lo.","Your battery is at 5 percent, use the power bank."\n50,hinglish,50,"Didi ki shaadi ki shopping ke liye hum Sarojini Nagar ja rahe hain.","We are going to Sarojini Nagar for my older sister\'s wedding shopping."\n51,hinglish,51,"Sir, please attendance mark kar dijiye, main bus mein phans gaya tha.","Sir, please mark my attendance, I got stuck in the bus."\n52,hinglish,52,"Aaj gym skip kar dete hain, kal se pakka regular jayenge.","Let\'s skip the gym today, from tomorrow we\'ll definitely go regularly."\n53,hinglish,53,"Ye PDF WhatsApp group pe forward kar do sabko.","Forward this PDF to everyone on the WhatsApp group."\n54,hinglish,54,"Mujhe samajh nahi aa raha ki physics ka ye numerical kaise solve hoga.","I don\'t understand how this physics numerical problem will be solved."\n55,hinglish,55,"Bhai, thoda dhaniya aur hari mirch free mein daal dena.","Brother, put in a little coriander and green chili for free."\n56,hinglish,56,"Weekend pe koi achhi series suggest karo binge-watch karne ke liye.","Suggest a good series to binge-watch on the weekend."\n58,hinglish,58,"Aunty ne pucha ki tumhare board exams kab khatam ho rahe hain.","Aunty asked when your board exams are ending."\n59,hinglish,59,"Main payment UPI se karunga, cash change nahi hai mere paas.","I will make the payment via UPI, I don\'t have cash change with me."\n62,hinglish,62,"Usne last moment pe ditch kar diya, ab poora plan chopat ho gaya!","They bailed at the last moment, now the whole plan is ruined!"\n64,hinglish,64,"Yaar, ye banda hamesha hawa mein baatein karta hai, serious mat lo.","Dude, this guy always talks big without substance, don\'t take him seriously."\n68,hinglish,68,"Chalo chhodo yaar, jo hua so hua, ab chill maaro aur biryani order karo.","Leave it buddy, whatever happened has happened, now chill out and order biryani."\n70,hinglish,70,"Mere dimaag ka dahi mat kar, pehle hi boss ne bahut dimaag kharab kar rakha hai.","Don\'t fry my brain, my boss has already messed with my head enough."\n71,hinglish,71,"Bhaiya, ye wala toh bahut mehnga lag raha hai, kuch toh discount laga do!","Brother, this one seems very expensive, give at least some discount!"\n73,hinglish,73,"Aaj toh subah se bhaag-daud lagi hui hai, ek cup kadak chai mil jaye toh maza aa jaye.","It\'s been non-stop running around since morning today, if I get one cup of strong tea it\'ll be amazing."\n75,hinglish,75,"Us bande ka attitude toh dekho, jaise khud ko CEO samajhta ho!","Look at that guy\'s attitude, as if he thinks himself to be the CEO!"\n76,hinglish,76,"Papa ne pocket money cut kar di kyunki pichle mahine bahut kharcha ho gaya tha.","Dad cut my pocket money because too much got spent last month."\n80,hinglish,80,"Chal bye, baad mein baat karte hain, mummy bula rahi hain khana khane ke liye.","Alright bye, we\'ll talk later, Mom is calling me to eat food."\n'

def sheet_csv_url(link):
    m = re.search(r"/spreadsheets/d/([\w-]+)", link)
    if not m:
        return link
    gid = re.search(r"[#&?]gid=(\d+)", link)
    return f"https://docs.google.com/spreadsheets/d/{m.group(1)}/export?format=csv" + (f"&gid={gid.group(1)}" if gid else "")

def load_sentences(folder=None):
    """The first test's sentences, or the sentences in `folder` (used for the second test)."""
    if folder is None and HINGLISH_SHEET.strip():
        src = sheet_csv_url(HINGLISH_SHEET.strip())
    else:
        src = os.path.join(folder or HINGLISH_DIR, "sentences.csv")
        if not os.path.exists(src):
            return None
    df = pd.read_csv(src)
    ren = {}
    for c in df.columns:                      # accept friendly column names
        k = str(c).strip().lower()
        if k in ("id", "number", "no", "no.", "#", "s.no", "sr"): name = "id"
        elif "hinglish" in k or k in ("text", "sentence", "say this"): name = "text"
        elif "english" in k: name = "english"
        elif "difficult" in k: name = "difficulty"
        elif k in ("type", "kind", "style"): name = "type"
        elif k in ("group", "set", "match"): name = "group"
        else: continue
        if name not in ren.values():
            ren[c] = name
    df = df.rename(columns=ren)
    if "text" not in df or "english" not in df:
        raise ValueError(f"sentences need a 'text' and an 'english' column. Found: {list(df.columns)}")
    if "id" not in df:
        df["id"] = range(1, len(df) + 1)
    df["id"] = pd.to_numeric(df["id"], errors="coerce")
    df = df.dropna(subset=["id", "text", "english"]).copy()
    df["id"] = df["id"].astype(int)
    df["type"] = df["type"].fillna("hinglish").astype(str).str.strip().str.lower() if "type" in df else "hinglish"
    if "group" not in df:
        df["group"] = df["id"]
    for c in ("text", "english"):
        df[c] = df[c].astype(str).str.strip()
    return df.drop_duplicates("id").set_index("id")

def eleven_request(url, body=None):
    req = urllib.request.Request(url, data=None if body is None else json.dumps(body).encode("utf8"),
                                 headers={"xi-api-key": ELEVENLABS_API_KEY, "Content-Type": "application/json"},
                                 method="GET" if body is None else "POST")
    try:
        with urllib.request.urlopen(req, timeout=120) as r:
            return r.read()
    except urllib.error.HTTPError as e:
        raise RuntimeError(f"ElevenLabs answered {e.code}: {e.read().decode('utf8', 'replace')[:400]}")

def eleven_left():
    """Characters left in the ElevenLabs quota, or None if the key may not read it."""
    try:
        s = json.loads(eleven_request("https://api.elevenlabs.io/v1/user/subscription"))
        return int(s["character_limit"]) - int(s["character_count"])
    except Exception:
        return None

def make_computer_voices(sent, folder=None):
    """Ask ElevenLabs to read each sentence. Saved as <id>_tts-<voice>.mp3, so they are never mistaken for people.
    One voice reads every sentence before the next voice starts, so a small quota still covers every sentence once."""
    global ELEVEN_MODEL
    adir = os.path.join(folder or HINGLISH_DIR, "audio")
    try:
        listed = json.loads(eleven_request("https://api.elevenlabs.io/v2/voices?page_size=100"))["voices"]
    except Exception:
        listed = json.loads(eleven_request("https://api.elevenlabs.io/v1/voices"))["voices"]
    # Free accounts may only use ElevenLabs' own default ("premade") voices through the API, so those go first.
    voices = sorted([v for v in listed if v.get("name")], key=lambda v: (v.get("category") != "premade", v["name"]))
    wanted = [w.strip().lower() for w in ELEVEN_VOICE_NAMES.split(",") if w.strip()]
    if wanted:
        voices = [v for v in voices if any(w in v["name"].lower() for w in wanted)]
    if not voices:
        raise RuntimeError("No ElevenLabs voice matched ELEVEN_VOICE_NAMES.")
    need, used, made, chars, refused = max(1, int(ELEVEN_VOICES)), [], 0, 0, 0
    for v in voices:
        if len(used) >= need or refused >= 12:
            break
        tag = "tts-" + (re.sub(r"[^a-z0-9]+", "", v["name"].lower())[:14] or v["voice_id"][:8].lower())
        url = f"https://api.elevenlabs.io/v1/text-to-speech/{v['voice_id']}?output_format=mp3_44100_128"
        made_here = 0
        try:
            for sid, row in sent.iterrows():
                path = os.path.join(adir, f"{int(sid):03d}_{tag}.mp3")
                if os.path.exists(path) and os.path.getsize(path) > 1000:
                    continue
                try:
                    mp3 = eleven_request(url, {"text": row.text, "model_id": ELEVEN_MODEL})
                except RuntimeError as e:
                    if "model" not in str(e).lower() or made or ELEVEN_MODEL == "eleven_multilingual_v2":
                        raise
                    print(f"{ELEVEN_MODEL} was refused ({str(e)[:200]}). Trying eleven_multilingual_v2 instead.")
                    ELEVEN_MODEL = "eleven_multilingual_v2"
                    mp3 = eleven_request(url, {"text": row.text, "model_id": ELEVEN_MODEL})
                with open(path, "wb") as f:
                    f.write(mp3)
                made += 1; made_here += 1; chars += len(row.text)
                time.sleep(0.3)
        except RuntimeError as e:
            if made_here == 0 and ("paid_plan_required" in str(e) or "library voices" in str(e)):
                refused += 1                      # this voice is not open to a free account: try the next one
                print(f"  voice '{v['name']}' needs a paid plan, skipped.")
                continue
            raise
        used.append(v["name"])
    if not used:
        raise RuntimeError("No voice on this account can be used through the API on the free plan.")
    print(f"Computer voices used: {', '.join(used)} (model {ELEVEN_MODEL}).")
    print(f"Made {made} new clips ({chars} characters of quota). Clips already saved were kept.")

_sent_path = os.path.join(HINGLISH_DIR, "sentences.csv")
if DEFAULT_SENTENCES and not HINGLISH_SHEET.strip() and not os.path.exists(_sent_path):
    with open(_sent_path, "w", encoding="utf8", newline="") as f:
        f.write(DEFAULT_SENTENCES)
    print("Wrote the sentence list that came with this notebook to", _sent_path)

_conf_path = os.path.join(CONFIRM_DIR, "sentences.csv")
if DEFAULT_CONFIRM and not os.path.exists(_conf_path):
    with open(_conf_path, "w", encoding="utf8", newline="") as f:
        f.write(DEFAULT_CONFIRM)
    print("Wrote the second test's sentence list to", _conf_path)

sent = load_sentences()
c_sent = load_sentences(CONFIRM_DIR)
print("Hinglish sentences, first test:", "none yet" if sent is None else f"{len(sent)} rows ({', '.join(f'{n} {t}' for t, n in sent.type.value_counts().items())})")
print("Hinglish sentences, second test:", "none yet" if c_sent is None else f"{len(c_sent)} rows")
if ELEVENLABS_API_KEY:
    left = eleven_left()
    print("ElevenLabs characters left:", "could not be read" if left is None else left)
    for _name, _s, _folder in (("first test", sent, None), ("second test", c_sent, CONFIRM_DIR)):
        if _s is None:
            continue
        print(f"Voices for the {_name}:")
        try:
            make_computer_voices(_s, _folder)
        except Exception as e:
            print("  Computer voices were NOT finished:", str(e)[:500])
            print("  The notebook carries on with whatever recordings already exist.")
elif sent is not None:
    print("No ElevenLabs key, so no computer voices are made. Human recordings in the audio folders are still used.")

## 2 · The pipeline

The next cell is the whole AI core, saved as `polyglot_core.py` so the live app can import the same code. Four functions matter:

- `transcribe(audio)` : speech to text (Whisper). It also reports which language it heard.
- `translate(text, src, tgt)` : text to text (NLLB-200, or Gemini as a comparison).
- `synthesize(text, lang)` : text to speech (MMS-TTS).
- `translate_speech(audio, targets)` : all three, once per listener language.

In [ ]:
%%writefile polyglot_core.py
"""
polyglot_core.py - Polyglot AI core (Part A)

Speech -> text -> translation -> speech, built on open pretrained models:

    speech recognition   OpenAI Whisper     openai/whisper-small
    translation          Meta NLLB-200      facebook/nllb-200-distilled-600M
    comparison engine    Google Gemini API  (optional, needs GEMINI_API_KEY)
    speech synthesis     Meta MMS-TTS       facebook/mms-tts-<lang>

Public API (the shapes are fixed in CONTRACT.md):

    transcribe(audio, lang_hint=None)              -> {"text", "lang", "mixed", "ms", ...}
    translate(text, src, tgt, engine="nllb")       -> {"text", "ms", ...}
    synthesize(text, lang)                         -> {"wav", "ms", ...}
    translate_speech(audio, targets, lang_hint)    -> {"original", "sourceLang", "mixed",
                                                       "results", "stageMs"}

Languages are the 2-letter codes the UI uses ("hi", "en", "ta", "bn", ...).
Only TESTED languages have measured results. The rest load on a best-effort basis.

Models load lazily on first use, on the GPU when there is one.
"""
from __future__ import annotations

import io
import json
import os
import re
import shutil
import subprocess
import tempfile
import threading
import time
import wave

__version__ = "0.1.0"

SR = 16000  # every model here works on 16 kHz mono audio

# UI code -> the code each model uses. "mms" is None where MMS-TTS has no voice.
LANGS = {
    "hi": {"name": "Hindi",      "nllb": "hin_Deva", "mms": "hin", "fleurs": "hi_in"},
    "en": {"name": "English",    "nllb": "eng_Latn", "mms": "eng", "fleurs": "en_us"},
    "bn": {"name": "Bengali",    "nllb": "ben_Beng", "mms": "ben", "fleurs": "bn_in"},
    "ta": {"name": "Tamil",      "nllb": "tam_Taml", "mms": "tam", "fleurs": "ta_in"},
    "te": {"name": "Telugu",     "nllb": "tel_Telu", "mms": "tel", "fleurs": "te_in"},
    "mr": {"name": "Marathi",    "nllb": "mar_Deva", "mms": "mar", "fleurs": "mr_in"},
    "kn": {"name": "Kannada",    "nllb": "kan_Knda", "mms": "kan", "fleurs": "kn_in"},
    "ml": {"name": "Malayalam",  "nllb": "mal_Mlym", "mms": "mal", "fleurs": "ml_in"},
    "gu": {"name": "Gujarati",   "nllb": "guj_Gujr", "mms": "guj", "fleurs": "gu_in"},
    "pa": {"name": "Punjabi",    "nllb": "pan_Guru", "mms": "pan", "fleurs": "pa_in"},
    "ur": {"name": "Urdu",       "nllb": "urd_Arab", "mms": "urd-script_arabic", "fleurs": "ur_pk"},
    "es": {"name": "Spanish",    "nllb": "spa_Latn", "mms": "spa", "fleurs": "es_419"},
    "fr": {"name": "French",     "nllb": "fra_Latn", "mms": "fra", "fleurs": "fr_fr"},
    "de": {"name": "German",     "nllb": "deu_Latn", "mms": "deu", "fleurs": "de_de"},
    "ar": {"name": "Arabic",     "nllb": "arb_Arab", "mms": "ara", "fleurs": "ar_eg"},
    "zh": {"name": "Chinese",    "nllb": "zho_Hans", "mms": None,  "fleurs": "cmn_hans_cn"},
    "ja": {"name": "Japanese",   "nllb": "jpn_Jpan", "mms": None,  "fleurs": "ja_jp"},
    "pt": {"name": "Portuguese", "nllb": "por_Latn", "mms": "por", "fleurs": "pt_br"},
    "it": {"name": "Italian",    "nllb": "ita_Latn", "mms": None,  "fleurs": "it_it"},
    "ko": {"name": "Korean",     "nllb": "kor_Hang", "mms": "kor", "fleurs": "ko_kr"},
}
TESTED = ["hi", "en", "ta", "bn"]

# Languages Whisper sometimes picks by mistake for our speakers. Only used so a
# wrong guess can still be translated instead of crashing.
_EXTRA_NLLB = {
    "ne": "npi_Deva", "sa": "san_Deva", "as": "asm_Beng", "sd": "snd_Arab",
    "si": "sin_Sinh", "ps": "pbt_Arab", "fa": "pes_Arab",
}
_MIXED_NAMES = {"hi": "Hinglish"}

CONFIG = {
    "asr_model": os.environ.get("POLYGLOT_ASR_MODEL", "openai/whisper-small"),
    "mt_model": os.environ.get("POLYGLOT_MT_MODEL", "facebook/nllb-200-distilled-600M"),
    "device": os.environ.get("POLYGLOT_DEVICE", ""),          # "" = GPU if present, else CPU
    # Languages auto-detect may choose from. None = the 20 UI languages, "all" = all of Whisper's.
    "candidates": None,
    # Code-mix flag. This default is a guess until the evaluation notebook calibrates it.
    "mix_threshold": float(os.environ.get("POLYGLOT_MIX_THRESHOLD", "0.15")),
    "latin_threshold": 0.20,
    # How speech is turned into another language. "cascade" = transcript -> NLLB.
    # "pivot" = Whisper's own speech-to-English, then NLLB from English.
    "route": {"default": os.environ.get("POLYGLOT_ROUTE", "cascade"),
              "mixed": os.environ.get("POLYGLOT_ROUTE_MIXED", "cascade")},
    "mt_beams": 4,
    "split_sentences": True,
    # Below this loudness a clip counts as silence (Whisper invents text on pure silence).
    # Keep it tiny: real recordings can be very quiet. FLEURS English is around 0.0004.
    "min_rms": 1e-4,
    "gemini_model": os.environ.get("GEMINI_MODEL", "gemini-2.5-flash"),
    "gemini_min_interval": 0.0,  # seconds between Gemini calls (raise it for free-tier batch runs)
}

_infer_lock = threading.RLock()
_asr: dict = {}
_mt: dict = {}
_tts: dict = {}


# --------------------------------------------------------------------------- audio

def wav_bytes(samples, sr: int = SR) -> bytes:
    """float samples in [-1, 1] -> 16-bit mono WAV bytes."""
    import numpy as np

    x = np.asarray(samples, dtype=np.float32).reshape(-1)
    pcm = (np.clip(x, -1.0, 1.0) * 32767.0).astype("<i2").tobytes()
    buf = io.BytesIO()
    with wave.open(buf, "wb") as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(int(sr))
        w.writeframes(pcm)
    return buf.getvalue()


def _read_wav_stdlib(data: bytes):
    import numpy as np

    with wave.open(io.BytesIO(data), "rb") as w:
        ch, width, sr, n = w.getnchannels(), w.getsampwidth(), w.getframerate(), w.getnframes()
        raw = w.readframes(n)
    if width == 2:
        x = np.frombuffer(raw, "<i2").astype(np.float32) / 32768.0
    elif width == 4:
        x = np.frombuffer(raw, "<i4").astype(np.float32) / 2147483648.0
    elif width == 1:
        x = (np.frombuffer(raw, np.uint8).astype(np.float32) - 128.0) / 128.0
    else:
        raise wave.Error(f"unsupported sample width {width}")
    if ch > 1:
        x = x[: len(x) // ch * ch].reshape(-1, ch).mean(axis=1)
    return x, sr


def _ffmpeg_decode(src) -> "np.ndarray":
    """Decode any audio (bytes or a file path) to 16 kHz mono float32 with ffmpeg."""
    import numpy as np

    exe = shutil.which("ffmpeg")
    if not exe:
        raise RuntimeError("This audio is not a plain WAV and ffmpeg is not installed to convert it.")
    tmp = None
    try:
        if isinstance(src, (bytes, bytearray, memoryview)):
            fd, tmp = tempfile.mkstemp(suffix=".audio")
            with os.fdopen(fd, "wb") as f:
                f.write(bytes(src))
            path = tmp
        else:
            path = os.fspath(src)
        cmd = [exe, "-nostdin", "-loglevel", "error", "-i", path,
               "-f", "f32le", "-ac", "1", "-ar", str(SR), "pipe:1"]
        p = subprocess.run(cmd, capture_output=True)
        if p.returncode != 0:
            raise RuntimeError("ffmpeg could not read the audio: " + p.stderr.decode("utf8", "replace")[-300:])
        return np.frombuffer(p.stdout, "<f4").copy()
    finally:
        if tmp and os.path.exists(tmp):
            os.remove(tmp)


def _resample(x, sr_from: int):
    import numpy as np

    if sr_from == SR or len(x) == 0:
        return x.astype(np.float32)
    try:
        from math import gcd
        from scipy.signal import resample_poly

        g = gcd(int(sr_from), SR)
        return resample_poly(x, SR // g, int(sr_from) // g).astype(np.float32)
    except ImportError:
        pass
    if shutil.which("ffmpeg"):
        return _ffmpeg_decode(wav_bytes(x, sr_from))
    n = int(round(len(x) * SR / sr_from))  # last resort: plain interpolation
    return np.interp(np.linspace(0, len(x) - 1, n), np.arange(len(x)), x).astype(np.float32)


def load_audio(src) -> "np.ndarray":
    """Anything audio-like -> float32 mono 16 kHz numpy array.

    Accepts WAV bytes, any encoded audio bytes (webm, m4a, mp3: needs ffmpeg), a file
    path, a numpy array already at 16 kHz, or a (sample_rate, array) pair.
    """
    import numpy as np

    if isinstance(src, tuple) and len(src) == 2:
        sr, arr = src
        arr = np.asarray(arr)
        scale = 32768.0 if arr.dtype.kind in "iu" else 1.0
        arr = arr.astype(np.float32) / scale
        if arr.ndim > 1:
            arr = arr.mean(axis=1)
        return _resample(arr, int(sr))
    if isinstance(src, np.ndarray):
        scale = 32768.0 if src.dtype.kind in "iu" else 1.0
        arr = src.astype(np.float32) / scale
        return arr.mean(axis=1) if arr.ndim > 1 else arr
    if isinstance(src, (str, os.PathLike)):
        with open(src, "rb") as f:
            data = f.read()
    else:
        data = bytes(src)
    if data[:4] == b"RIFF":
        try:
            x, sr = _read_wav_stdlib(data)
            return _resample(x, sr)
        except Exception:
            pass
    try:
        import soundfile as sf

        x, sr = sf.read(io.BytesIO(data), dtype="float32", always_2d=True)
        return _resample(x.mean(axis=1), sr)
    except Exception:
        pass
    return _ffmpeg_decode(data)


# --------------------------------------------------------------------------- models

def _quiet():
    """Hide transformers' harmless generate() warnings so notebook output stays readable."""
    try:
        from transformers.utils import logging as hf_logging

        hf_logging.set_verbosity_error()
    except Exception:
        pass


def _device() -> str:
    if CONFIG.get("device"):
        return CONFIG["device"]
    import torch

    return "cuda" if torch.cuda.is_available() else "cpu"


def info() -> dict:
    import torch
    import transformers

    dev = _device()
    return {
        "core": __version__,
        "asr_model": CONFIG["asr_model"],
        "mt_model": CONFIG["mt_model"],
        "tts_model": "facebook/mms-tts-<lang>",
        "device": dev,
        "gpu": torch.cuda.get_device_name(0) if dev == "cuda" else None,
        "torch": torch.__version__,
        "transformers": transformers.__version__,
    }


def _load_asr() -> dict:
    name = CONFIG["asr_model"]
    if _asr.get("name") == name:
        return _asr
    import torch
    from transformers import WhisperForConditionalGeneration, WhisperProcessor

    _quiet()

    with _infer_lock:
        if _asr:  # switching models: free the old one first, so two never sit on the GPU together
            import gc

            _asr.clear()
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        proc = WhisperProcessor.from_pretrained(name)
        model = WhisperForConditionalGeneration.from_pretrained(name)
        dev = _device()
        model = model.to(dev)
        if dev == "cuda":
            model = model.half()
        model.eval()
        lang_to_id = getattr(model.generation_config, "lang_to_id", None)
        if not lang_to_id:  # English-only or unusual checkpoint
            from transformers.models.whisper.tokenization_whisper import LANGUAGES

            lang_to_id = {f"<|{c}|>": proc.tokenizer.convert_tokens_to_ids(f"<|{c}|>") for c in LANGUAGES}
        codes = [k.strip("<|>") for k in lang_to_id]
        ids = torch.tensor(list(lang_to_id.values()), dtype=torch.long, device=dev)
        _asr.clear()
        _asr.update(name=name, proc=proc, model=model, device=dev, codes=codes, ids=ids,
                    index={c: i for i, c in enumerate(codes)})
    return _asr


def set_asr_model(name: str) -> dict:
    """Switch the speech-to-text model, for example to compare a bigger Whisper."""
    CONFIG["asr_model"] = name
    _load_asr()
    return info()


def _load_mt() -> dict:
    name = CONFIG["mt_model"]
    if _mt.get("name") == name:
        return _mt
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

    _quiet()

    with _infer_lock:
        tok = AutoTokenizer.from_pretrained(name)
        model = AutoModelForSeq2SeqLM.from_pretrained(name)
        dev = _device()
        model = model.to(dev)
        if dev == "cuda":
            model = model.half()
        model.eval()
        _mt.clear()
        _mt.update(name=name, tok=tok, model=model, device=dev)
    return _mt


def _load_tts(lang: str):
    """MMS-TTS voice for one language, or None when there isn't one."""
    if lang in _tts:
        return _tts[lang]
    code = LANGS.get(lang, {}).get("mms")
    entry = None
    if code:
        try:
            from transformers import AutoTokenizer, VitsModel

            _quiet()

            with _infer_lock:
                name = f"facebook/mms-tts-{code}"
                tok = AutoTokenizer.from_pretrained(name)
                model = VitsModel.from_pretrained(name).to(_device()).eval()
                entry = {"name": name, "tok": tok, "model": model}
        except Exception as e:  # no voice for this language, or no internet
            print(f"[polyglot] no MMS-TTS voice for '{lang}': {type(e).__name__}: {str(e)[:120]}")
    _tts[lang] = entry
    return entry


def warmup(langs=("hi", "en"), tts: bool = True) -> dict:
    """Load the models now, so the first real request is not slow."""
    import numpy as np

    t0 = time.perf_counter()
    _load_asr()
    _load_mt()
    tone = (0.1 * np.sin(2 * np.pi * 220 * np.arange(SR) / SR)).astype(np.float32)
    transcribe_batch([tone], lang_hint="en", with_probs=True)
    translate_batch(["hello"], "en", "hi")
    if tts:
        for lang in langs:
            synthesize("ok", lang)
    return {**info(), "warmup_s": round(time.perf_counter() - t0, 1)}


# --------------------------------------------------------------------------- speech -> text

def _features(arrays):
    a = _asr
    fe = a["proc"].feature_extractor
    out = fe(arrays, sampling_rate=SR, return_tensors="pt", return_attention_mask=True)
    model = a["model"]
    feats = out["input_features"].to(device=a["device"], dtype=model.dtype)
    mask = out.get("attention_mask")
    if mask is not None:
        mask = mask.to(a["device"])
    return feats, mask


def _lang_probs(feats):
    """Whisper's own language guess: one decoder step, softmax over its language tokens."""
    import torch

    model = _asr["model"]
    with torch.no_grad():
        enc = model.get_encoder()(feats)
        start = model.config.decoder_start_token_id
        dec = torch.full((feats.shape[0], 1), start, dtype=torch.long, device=feats.device)
        logits = model(encoder_outputs=enc, decoder_input_ids=dec).logits[:, -1, :].float()
        probs = torch.softmax(logits[:, _asr["ids"]], dim=-1)
    return probs.cpu().numpy()


def _candidate_codes(candidates):
    if candidates is None:
        candidates = CONFIG.get("candidates")
    if candidates == "all":
        return None
    return list(candidates) if candidates else list(LANGS)


def _pick_lang(prob_row, candidates) -> str:
    codes = _asr["codes"]
    allowed = _candidate_codes(candidates)
    if allowed is None:
        return codes[int(prob_row.argmax())]
    best, best_p = None, -1.0
    for c in allowed:
        i = _asr["index"].get(c)
        if i is not None and prob_row[i] > best_p:
            best, best_p = c, float(prob_row[i])
    return best or codes[int(prob_row.argmax())]


def latin_ratio(text: str) -> float:
    """Share of words written in Latin letters (English words left inside Hindi text)."""
    words = re.findall(r"[^\W\d_]+", text or "")
    if not words:
        return 0.0
    return sum(1 for w in words if re.search(r"[A-Za-z]", w)) / len(words)


def mix_score(probs: dict) -> float:
    """0 = clearly one language, 0.5 = Hindi and English equally likely."""
    p_hi, p_en = probs.get("hi", 0.0), probs.get("en", 0.0)
    return min(p_hi, p_en) / (p_hi + p_en + 1e-9)


def _mix_flag(lang: str, probs: dict, text: str):
    name = _MIXED_NAMES.get(lang)
    if not name:
        return None
    if mix_score(probs) >= CONFIG["mix_threshold"] or latin_ratio(text) >= CONFIG["latin_threshold"]:
        return name
    return None


def _generate(feats, mask, lang: str, task: str):
    import torch

    a = _asr
    kw = {"language": lang, "task": task}
    if mask is not None:
        kw["attention_mask"] = mask
    with torch.no_grad():
        ids = a["model"].generate(feats, **kw)
    texts = a["proc"].batch_decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    return [t.strip() for t in texts]


def detect_language_batch(audios, batch_size: int = 16) -> list:
    """Whisper's language probabilities for each clip: [{"hi": 0.91, "ur": 0.05, ...}, ...]."""
    a = _load_asr()
    arrays = [load_audio(x) for x in audios]
    out = []
    for s in range(0, len(arrays), batch_size):
        with _infer_lock:
            feats, _ = _features(arrays[s:s + batch_size])
            probs = _lang_probs(feats)
        for row in probs:
            out.append({c: float(p) for c, p in zip(a["codes"], row)})
    return out


def transcribe_batch(audios, lang_hint=None, candidates=None, task: str = "transcribe",
                     batch_size: int = 16, with_probs: bool = True) -> list:
    """Transcribe many clips.

    lang_hint   None (detect), one code for all clips, or a list with one code (or None) per clip.
    candidates  languages detection may pick from (see CONFIG["candidates"]).
    task        "transcribe", or "translate" for Whisper's built-in speech -> English.
    """
    import numpy as np

    a = _load_asr()
    arrays = [load_audio(x) for x in audios]
    n = len(arrays)
    hints = list(lang_hint) if isinstance(lang_hint, (list, tuple)) else [lang_hint] * n
    for h in hints:
        if h is not None and h not in a["index"]:
            raise ValueError(f"Whisper has no language '{h}'")
    out = [None] * n
    for s in range(0, n, batch_size):
        idx = list(range(s, min(n, s + batch_size)))
        t0 = time.perf_counter()
        silent = [len(arrays[i]) < 0.25 * SR or float(np.sqrt(np.mean(arrays[i] ** 2))) < CONFIG["min_rms"]
                  for i in idx]
        with _infer_lock:
            feats, mask = _features([arrays[i] for i in idx])
            need_probs = with_probs or any(hints[i] is None for i in idx)
            probs = _lang_probs(feats) if need_probs else None
            langs = [hints[i] or _pick_lang(probs[j], candidates) for j, i in enumerate(idx)]
            texts = [""] * len(idx)
            for lang in sorted(set(langs)):
                rows = [j for j, l in enumerate(langs) if l == lang and not silent[j]]
                if not rows:
                    continue
                decoded = _generate(feats[rows], mask[rows] if mask is not None else None, lang, task)
                for j, t in zip(rows, decoded):
                    texts[j] = t
        ms = int(round((time.perf_counter() - t0) * 1000 / len(idx)))
        for j, i in enumerate(idx):
            item = {"text": texts[j], "lang": langs[j], "mixed": None, "ms": ms}
            if probs is not None:
                pd_ = {c: float(p) for c, p in zip(a["codes"], probs[j])}
                top = sorted(pd_.items(), key=lambda kv: -kv[1])[:5]
                item["langProbs"] = {c: round(p, 4) for c, p in top}
                item["mixScore"] = round(mix_score(pd_), 4)
                item["latinRatio"] = round(latin_ratio(texts[j]), 3)
                if task == "transcribe":
                    item["mixed"] = _mix_flag(langs[j], pd_, texts[j])
            out[i] = item
    return out


def transcribe(audio, lang_hint=None, candidates=None) -> dict:
    """Speech -> text.  audio: 16 kHz mono WAV bytes (other formats are converted)."""
    t0 = time.perf_counter()
    r = transcribe_batch([audio], lang_hint=lang_hint, candidates=candidates)[0]
    r["ms"] = int(round((time.perf_counter() - t0) * 1000))
    return r


# --------------------------------------------------------------------------- text -> text

def nllb_code(lang: str):
    if lang in LANGS:
        return LANGS[lang]["nllb"]
    return _EXTRA_NLLB.get(lang)


def _split_sentences(text: str) -> list:
    """NLLB was trained on single sentences and can drop later ones, so feed it one at a time."""
    parts = [p.strip() for p in re.split(r"(?<=[.!?।॥])\s+", text.strip()) if p.strip()]
    merged = []
    for p in parts:
        if merged and len(merged[-1]) < 12:
            merged[-1] += " " + p
        else:
            merged.append(p)
    return merged or [text.strip()]


def _nllb_translate(texts: list, src: str, tgt: str, batch_size: int) -> list:
    import torch

    m = _load_mt()
    tok, model = m["tok"], m["model"]
    out = [""] * len(texts)
    order = sorted((i for i, t in enumerate(texts) if t.strip()), key=lambda i: len(texts[i]))
    with _infer_lock:
        tok.src_lang = nllb_code(src)
        bos = tok.convert_tokens_to_ids(nllb_code(tgt))
        for s in range(0, len(order), batch_size):
            chunk = order[s:s + batch_size]
            enc = tok([texts[i] for i in chunk], return_tensors="pt", padding=True,
                      truncation=True, max_length=256).to(m["device"])
            with torch.no_grad():
                gen = model.generate(**enc, forced_bos_token_id=bos,
                                     num_beams=CONFIG["mt_beams"], max_length=256)
            for i, t in zip(chunk, tok.batch_decode(gen, skip_special_tokens=True)):
                out[i] = t.strip()
    return out


_gem_last = [0.0]


def _gemini_request(method: str, url: str, body=None, timeout: int = 90):
    import urllib.error
    import urllib.request

    key = os.environ.get("GEMINI_API_KEY") or CONFIG.get("gemini_api_key")
    if not key:
        raise RuntimeError("GEMINI_API_KEY is not set")
    data = json.dumps(body).encode("utf8") if body is not None else None
    last = None
    for wait in (0, 6, 15, 30, 60):
        if wait:
            time.sleep(wait)
        gap = CONFIG["gemini_min_interval"] - (time.time() - _gem_last[0])
        if gap > 0:
            time.sleep(gap)
        _gem_last[0] = time.time()
        req = urllib.request.Request(url, data=data, method=method, headers={
            "Content-Type": "application/json", "x-goog-api-key": key})
        try:
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return json.loads(r.read().decode("utf8"))
        except urllib.error.HTTPError as e:
            detail = e.read().decode("utf8", "replace")[:300]
            last = RuntimeError(f"Gemini HTTP {e.code}: {detail}")
            last.code = e.code
            if e.code not in (429, 500, 502, 503, 504):
                raise last
        except Exception as e:  # network hiccup: retry
            last = RuntimeError(f"Gemini request failed: {type(e).__name__}: {e}")
    raise last


_GEMINI_BASE = "https://generativelanguage.googleapis.com/v1beta"


def _gemini_pick_model() -> str:
    """Find a current Flash model if the configured one no longer exists."""
    data = _gemini_request("GET", f"{_GEMINI_BASE}/models?pageSize=200")
    names = [m["name"].split("/")[-1] for m in data.get("models", [])
             if "generateContent" in m.get("supportedGenerationMethods", [])]
    skip = ("tts", "image", "live", "audio", "embedding", "exp", "preview", "lite", "8b")
    good = [n for n in names if "flash" in n and not any(s in n for s in skip)]
    if not good:
        good = [n for n in names if "flash" in n] or names
    if not good:
        raise RuntimeError("No usable Gemini model found for this API key")
    return sorted(good)[-1]


def gemini_generate(prompt: str, json_mode: bool = False) -> str:
    cfg = {"temperature": 0}
    if json_mode:  # ask for a JSON array of strings and nothing else
        cfg["responseMimeType"] = "application/json"
        cfg["responseSchema"] = {"type": "ARRAY", "items": {"type": "STRING"}}
    body = {"contents": [{"parts": [{"text": prompt}]}], "generationConfig": cfg}
    try:
        data = _gemini_request("POST", f"{_GEMINI_BASE}/models/{CONFIG['gemini_model']}:generateContent", body)
    except RuntimeError as e:
        if getattr(e, "code", None) != 404:
            raise
        CONFIG["gemini_model"] = _gemini_pick_model()
        data = _gemini_request("POST", f"{_GEMINI_BASE}/models/{CONFIG['gemini_model']}:generateContent", body)
    cands = data.get("candidates") or []
    if not cands:
        raise RuntimeError("Gemini returned no answer: " + json.dumps(data)[:200])
    parts = cands[0].get("content", {}).get("parts", [])
    return "".join(p.get("text", "") for p in parts if not p.get("thought")).strip()


def _gemini_translate(texts: list, src: str, tgt: str, chunk: int = 20) -> list:
    s_name = LANGS.get(src, {}).get("name", src)
    t_name = LANGS.get(tgt, {}).get("name", tgt)
    head = (
        f"You are a professional translator. Translate each numbered item from {s_name} into {t_name}.\n"
        f"The source can mix {s_name} with English words (code-switching) and can come from speech "
        "recognition, so it may contain small errors. Translate the meaning naturally and completely.\n"
    )
    out = [""] * len(texts)
    todo = [i for i, t in enumerate(texts) if t.strip()]
    for s in range(0, len(todo), chunk):
        part = todo[s:s + chunk]
        items = "\n".join(f"{k + 1}. {texts[i]}" for k, i in enumerate(part))
        prompt = (head + f"Return ONLY a JSON array of exactly {len(part)} strings, in the same order, "
                         "with no explanations.\n\n" + items)
        got = None
        try:
            got = json.loads(gemini_generate(prompt, json_mode=True))
        except (ValueError, RuntimeError) as e:
            if "GEMINI_API_KEY" in str(e):
                raise
        if isinstance(got, dict) and len(got) == 1:      # {"translations": [...]}
            got = next(iter(got.values()))
        if isinstance(got, list) and len(got) == len(part):
            for i, t in zip(part, got):
                out[i] = str(t).strip()
        else:  # count mismatch or bad JSON: go one by one
            for i in part:
                out[i] = gemini_generate(
                    head + f"Return only the {t_name} translation, nothing else.\n\n{texts[i]}").strip()
    return out


def translate_batch(texts, src: str, tgt: str, engine: str = "nllb", batch_size: int = 16) -> list:
    """Translate a list of texts. Returns a list of strings in the same order."""
    texts = [str(t or "") for t in texts]
    if src == tgt:
        return texts
    if engine == "gemini":
        return _gemini_translate(texts, src, tgt)
    if engine != "nllb":
        raise ValueError(f"unknown engine '{engine}'")
    if nllb_code(src) is None or nllb_code(tgt) is None:
        return texts  # a language NLLB is not set up for here: leave the text as it is
    if not CONFIG["split_sentences"]:
        return _nllb_translate(texts, src, tgt, batch_size)
    flat, owner = [], []
    for i, t in enumerate(texts):
        for piece in (_split_sentences(t) if t.strip() else []):
            flat.append(piece)
            owner.append(i)
    done = _nllb_translate(flat, src, tgt, batch_size)
    out = [[] for _ in texts]
    for i, t in zip(owner, done):
        out[i].append(t)
    return [" ".join(p) for p in out]


def translate(text: str, src: str, tgt: str, engine: str = "nllb") -> dict:
    """Text -> text.  engine: "nllb" (main) or "gemini" (comparison, falls back to NLLB)."""
    t0 = time.perf_counter()
    note = None
    try:
        out = translate_batch([text], src, tgt, engine=engine)[0]
    except Exception as e:
        if engine == "nllb":
            raise
        note = f"{engine} failed ({type(e).__name__}: {str(e)[:120]}), used nllb"
        engine = "nllb"
        out = translate_batch([text], src, tgt, engine="nllb")[0]
    r = {"text": out, "ms": int(round((time.perf_counter() - t0) * 1000)), "engine": engine}
    if note:
        r["note"] = note
    return r


# --------------------------------------------------------------------------- text -> speech

def synthesize(text: str, lang: str) -> dict:
    """Text -> speech.  Returns {"wav": WAV bytes, "ms"}. "wav" is b"" when the language has no voice."""
    import numpy as np
    import torch

    t0 = time.perf_counter()
    text = (text or "").strip()
    entry = _load_tts(lang) if text else None
    if not entry:
        return {"wav": b"", "ms": 0}
    tok, model = entry["tok"], entry["model"]
    if getattr(tok, "is_uroman", False):
        try:
            import uroman

            text = uroman.Uroman().romanize_string(text)
        except Exception:
            return {"wav": b"", "ms": 0}
    sr = int(model.config.sampling_rate)
    chunks = []
    with _infer_lock:
        for piece in _split_sentences(text):
            enc = tok(piece, return_tensors="pt")
            if enc["input_ids"].shape[-1] == 0:
                continue
            enc = {k: v.to(model.device) for k, v in enc.items()}
            with torch.no_grad():
                wave_t = model(**enc).waveform[0]
            chunks.append(wave_t.float().cpu().numpy())
            chunks.append(np.zeros(int(0.12 * sr), dtype=np.float32))
    if not chunks:
        return {"wav": b"", "ms": 0}
    audio = np.concatenate(chunks[:-1])
    return {"wav": wav_bytes(audio, sr), "ms": int(round((time.perf_counter() - t0) * 1000)), "sr": sr}


# --------------------------------------------------------------------------- speech -> speech

def translate_speech(audio, targets, lang_hint=None, engine: str = "nllb", candidates=None,
                     tts: bool = True, route=None) -> dict:
    """One utterance in, one translation per target language out.

    Transcribes once, then translates and speaks once per distinct target. A target equal
    to the speaker's language gets the original text and no audio.
    """
    arr = load_audio(audio)
    targets = list(dict.fromkeys(targets))

    t0 = time.perf_counter()
    asr = transcribe_batch([arr], lang_hint=lang_hint, candidates=candidates)[0]
    asr_ms = (time.perf_counter() - t0) * 1000
    src, original, mixed = asr["lang"], asr["text"], asr["mixed"]
    route = route or CONFIG["route"]["mixed" if mixed else "default"]

    t0 = time.perf_counter()
    texts = {}
    others = [t for t in targets if t != src]
    if original and others:
        if route == "pivot" and src != "en":
            english = transcribe_batch([arr], lang_hint=src, task="translate", with_probs=False)[0]["text"]
            for t in others:
                texts[t] = english if t == "en" else translate_batch([english], "en", t, engine=engine)[0]
        else:
            for t in others:
                texts[t] = translate(original, src, t, engine=engine)["text"]
    mt_ms = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    results = {}
    for t in targets:
        if t == src:
            results[t] = {"text": original, "wav": b""}
        else:
            text = texts.get(t, "")
            results[t] = {"text": text, "wav": synthesize(text, t)["wav"] if (tts and text) else b""}
    tts_ms = (time.perf_counter() - t0) * 1000

    return {
        "original": original,
        "sourceLang": src,
        "mixed": mixed,
        "results": results,
        "stageMs": {"asr": int(round(asr_ms)), "mt": int(round(mt_ms)), "tts": int(round(tts_ms))},
        "route": route,
        "langProbs": asr.get("langProbs", {}),
    }


# --------------------------------------------------------------------------- command line

if __name__ == "__main__":
    import argparse

    ap = argparse.ArgumentParser(description="Polyglot: translate one audio file.")
    ap.add_argument("audio", help="audio file (wav, m4a, mp3, webm, ...)")
    ap.add_argument("--to", nargs="+", default=["en"], help="target language codes, e.g. en ta")
    ap.add_argument("--lang", default=None, help="language being spoken, if known (e.g. hi)")
    ap.add_argument("--engine", default="nllb", choices=["nllb", "gemini"])
    ap.add_argument("--save", default=None, help="folder to save the translated speech into")
    args = ap.parse_args()

    res = translate_speech(args.audio, args.to, lang_hint=args.lang, engine=args.engine)
    print(json.dumps({k: v for k, v in res.items() if k != "results"}, ensure_ascii=False, indent=2))
    for code, r in res["results"].items():
        print(f"[{code}] {r['text']}   ({len(r['wav'])} bytes of audio)")
        if args.save and r["wav"]:
            os.makedirs(args.save, exist_ok=True)
            with open(os.path.join(args.save, f"out_{code}.wav"), "wb") as f:
                f.write(r["wav"])

In [ ]:
#@title 3 · Load the models
import polyglot_core as pc
importlib.reload(pc)

pc.CONFIG["asr_model"] = ASR_MODEL
pc.CONFIG["min_rms"] = 0.0   # score every test clip. FLEURS English is recorded very quietly, and a loudness check would drop it.
if GEMINI_API_KEY:
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
    pc.CONFIG["gemini_min_interval"] = 4.5   # stay under the free tier's requests-per-minute limit

INFO = pc.warmup(langs=("hi", "en"))
print(json.dumps(INFO, indent=2))
if INFO["device"] != "cuda":
    print("\nNO GPU FOUND, so this will be slow. Use Runtime > Change runtime type > T4 GPU.")
    if not QUICK_TEST and "POLYGLOT_N" not in os.environ:
        N_PER_LANG = min(N_PER_LANG, 10)
        print("Using only", N_PER_LANG, "sentences per language so the run can finish.")

## 4 · Test data: the same sentences in four languages

We use **FLEURS** (Google), a public set of short sentences read aloud by native speakers in 102 languages. The sentences are professional translations of each other, so sentence 1234 in Hindi means the same as sentence 1234 in Tamil.

We pick sentences at random (fixed seed) from the **test split**, and use the *same* sentences in every language. Any difference between languages then comes from the language, not from easier or harder sentences. None of our models were trained on this data.

In [ ]:
#@title 4 · Download the test data
import tarfile
import numpy as np, pandas as pd
from huggingface_hub import hf_hub_download
pd.set_option("display.max_colwidth", 220); pd.set_option("display.width", 250)

FLEURS_REPO = "google/fleurs"

def _unquote(t):
    """Some FLEURS rows are wrapped in spreadsheet-style quotes ("a ""b"" c"). Undo that."""
    t = t.strip()
    if len(t) > 1 and t[0] == '"' and t[-1] == '"' and '""' in t:
        t = t[1:-1].replace('""', '"')
    return t

def fleurs_meta(lang, split="test"):
    """One row per sentence: id -> audio file name and the written sentence."""
    cfg = pc.LANGS[lang]["fleurs"]
    path = hf_hub_download(FLEURS_REPO, f"data/{cfg}/{split}.tsv", repo_type="dataset")
    rows = []
    with open(path, encoding="utf8") as f:
        for line in f:
            p = line.rstrip("\n").split("\t")
            if len(p) >= 3 and p[0].strip().isdigit():
                rows.append((int(p[0]), p[1], _unquote(p[2])))
    return pd.DataFrame(rows, columns=["id", "file", "text"]).drop_duplicates("id").set_index("id")

def fleurs_audio(lang, meta, ids, split="test"):
    """Read only the clips we need from the archive: {sentence id: waveform at 16 kHz}."""
    cfg = pc.LANGS[lang]["fleurs"]
    path = hf_hub_download(FLEURS_REPO, f"data/{cfg}/audio/{split}.tar.gz", repo_type="dataset")
    want = {meta.loc[s, "file"]: s for s in ids}
    out = {}
    with tarfile.open(path, "r:gz") as tar:
        for m in tar:
            name = os.path.basename(m.name)
            if m.isfile() and name in want:
                out[want[name]] = pc.load_audio(tar.extractfile(m).read())
                if len(out) == len(want):
                    break
    return out

t0 = time.time()
fl = {l: fleurs_meta(l) for l in LANGS_TESTED}
common = sorted(set.intersection(*[set(m.index) for m in fl.values()]))
random.Random(SEED).shuffle(common)
SIDS = sorted(common[:N_PER_LANG])
text = {l: {s: str(fl[l].loc[s, "text"]) for s in SIDS} for l in LANGS_TESTED}
audio = {l: fleurs_audio(l, fl[l], SIDS) for l in LANGS_TESTED}
assert all(len(audio[l]) == len(SIDS) for l in LANGS_TESTED), "some clips are missing"

print(f"{len(common)} sentences exist in all {len(LANGS_TESTED)} languages. Using {len(SIDS)} of them.")
for l in LANGS_TESTED:
    mins = sum(len(a) for a in audio[l].values()) / pc.SR / 60
    print(f"  {pc.LANGS[l]['name']:8s} {len(audio[l])} clips, {mins:.1f} minutes of speech")
print(f"Loaded in {time.time() - t0:.0f} s.\n\nThe same sentence in each language:")
for l in LANGS_TESTED:
    print(f"  [{l}] {text[l][SIDS[0]]}")

In [ ]:
#@title 5 · One example, start to finish
from IPython.display import Audio, display

sid = SIDS[0]
clip = audio["hi"][sid]
print("INPUT: a Hindi speaker says this.")
display(Audio(clip, rate=pc.SR))

res = pc.translate_speech(clip, ["en", "ta"], lang_hint="hi")
print("Polyglot heard :", res["original"])
print("Actually said  :", text["hi"][sid])
for c in ["en", "ta"]:
    out = res["results"][c]
    print(f"\nOUTPUT in {pc.LANGS[c]['name']}: {out['text']}")
    print(f"A human translator wrote: {text[c][sid]}")
    if out["wav"]:
        display(Audio(out["wav"]))
print("\nTime per step (ms):", res["stageMs"])

## 6 · How we score

- **Speech to text: word error rate (WER).** The share of words that are wrong, missing or extra. 0% is perfect. We also give the character error rate (CER), which is fairer for languages with long joined-up words such as Tamil.
- **Translation: chrF.** How much of the human translation's characters and word pieces the model's translation also has. 0 to 100, higher is better.
- **Every score comes with a 95% range**, from re-sampling the test sentences. If two ranges overlap a lot, the difference may just be luck.

Before scoring we lower-case the text and strip punctuation. We wrote our own cleaner because Whisper's standard one deletes the vowel signs of Indian scripts, which makes the scores wrong.

In [ ]:
#@title 6 · Scoring functions
import jiwer, sacrebleu
from rapidfuzz.distance import Levenshtein

def norm_text(s):
    """Lower-case, drop punctuation, keep letters, digits and the vowel signs of Indian scripts."""
    s = unicodedata.normalize("NFKC", str(s if s is not None else "")).lower()
    out = []
    for ch in s:
        cat = unicodedata.category(ch)
        if cat[0] in "LMN":
            out.append(ch)
        elif ch in "'’" or cat == "Cf":
            continue                      # don't -> dont ; invisible joiners
        else:
            out.append(" ")
    return " ".join("".join(out).split())

def wer_parts(ref, hyp):
    r, h = norm_text(ref).split(), norm_text(hyp).split()
    return Levenshtein.distance(r, h), len(r)

def cer_parts(ref, hyp):
    r, h = norm_text(ref), norm_text(hyp)
    return Levenshtein.distance(r, h), len(r)

def chrf(hyps, refs):
    return sacrebleu.corpus_chrf(list(hyps), [list(refs)]).score

def chrf1(hyp, ref):
    return sacrebleu.sentence_chrf(str(hyp), [str(ref)]).score

def boot_ci(n, stat, n_boot=300, seed=SEED):
    """95% range of stat(idx) when the n test items are re-sampled with replacement."""
    r = np.random.default_rng(seed)
    vals = [stat(r.integers(0, n, n)) for _ in range(n_boot)]
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def chrf_ci(hyps, refs, n_boot=100):
    hyps, refs = list(hyps), list(refs)
    lo, hi = boot_ci(len(refs), lambda i: chrf([hyps[j] for j in i], [refs[j] for j in i]), n_boot)
    return chrf(hyps, refs), lo, hi

def chrf_gain_ci(base, new, refs, n_boot=100):
    """How much better `new` is than `base` on the same sentences (chrF points) with its 95% range."""
    base, new, refs = list(base), list(new), list(refs)
    f = lambda i: chrf([new[j] for j in i], [refs[j] for j in i]) - chrf([base[j] for j in i], [refs[j] for j in i])
    lo, hi = boot_ci(len(refs), f, n_boot)
    return chrf(new, refs) - chrf(base, refs), lo, hi

def pick(probs, allowed=None):
    """Most likely language, optionally only among `allowed`."""
    items = probs.items() if allowed is None else [(c, probs.get(c, 0.0)) for c in allowed]
    return max(items, key=lambda kv: kv[1])[0]

print(norm_text("Don't STOP!  कल मीटिंग है, ठीक?"), "|", wer_parts("a b c d", "a x c"), "|", round(chrf1("hello there", "hello here"), 1))

## 7 · Speech to text, language by language

Whisper has to decide which language it is hearing before it writes anything. We compare three ways of deciding:

1. **Baseline: auto.** Whisper guesses among all 99 languages it knows. This is what you get out of the box.
2. **Room.** Whisper may only choose a language that someone in the call speaks (here: the four we test).
3. **Hint.** Each speaker tells the app their language once, when they join. This is what Polyglot does.

In [ ]:
#@title 7 · Speech-to-text test
import matplotlib.pyplot as plt

lid = {}                                   # (lang, sentence) -> Whisper's language probabilities
for l in LANGS_TESTED:
    for s, p in zip(SIDS, pc.detect_language_batch([audio[l][s] for s in SIDS])):
        lid[(l, s)] = p

COND = {"auto": lambda l, p: pick(p), "room": lambda l, p: pick(p, LANGS_TESTED), "hint": lambda l, p: l}
COND_LABEL = {"auto": "Baseline: auto (99 languages)", "room": "Room (4 languages)", "hint": "Hint (speaker's language)"}

asr_cache = {}                             # (lang, sentence, language Whisper was told to write) -> transcript
def transcribe_forced(keys):
    todo = {}
    for k in dict.fromkeys(keys):
        if k not in asr_cache:
            todo.setdefault(k[2], []).append(k)
    for forced, ks in todo.items():
        outs = pc.transcribe_batch([audio[l][s] for l, s, _ in ks], lang_hint=forced, with_probs=False)
        for k, o in zip(ks, outs):
            asr_cache[k] = o["text"]

t0 = time.time()
transcribe_forced([(l, s, fn(l, lid[(l, s)])) for l in LANGS_TESTED for s in SIDS for fn in COND.values()])

rows = []
for l in LANGS_TESTED:
    for s in SIDS:
        for c, fn in COND.items():
            used = fn(l, lid[(l, s)])
            hyp = asr_cache[(l, s, used)]
            e, n = wer_parts(text[l][s], hyp)
            ce, cn = cer_parts(text[l][s], hyp)
            rows.append(dict(lang=l, sid=s, cond=c, used_lang=used, ref=text[l][s], hyp=hyp, werr=e, wn=n, cerr=ce, cn=cn))
asr_df = pd.DataFrame(rows)

def asr_table(df):
    out = []
    for (l, c), g in df.groupby(["lang", "cond"], sort=False):
        e, n = g.werr.to_numpy(), g.wn.to_numpy()
        lo, hi = boot_ci(len(g), lambda i: e[i].sum() / max(n[i].sum(), 1))
        out.append({"lang": l, "cond": c, "Language": pc.LANGS[l]["name"], "Setting": COND_LABEL[c],
                    "WER %": 100 * e.sum() / max(n.sum(), 1), "WER 95% range": f"{100 * lo:.1f} to {100 * hi:.1f}",
                    "CER %": 100 * g.cerr.sum() / max(g.cn.sum(), 1),
                    "Right language %": 100 * float((g.used_lang == g.lang).mean()), "Clips": len(g)})
    return pd.DataFrame(out)

asr_sum = asr_table(asr_df)
print(f"Done in {time.time() - t0:.0f} s. Lower WER is better.\n")
display(asr_sum.drop(columns=["lang", "cond"]).round(1))

g = asr_df[(asr_df.lang == "en") & (asr_df.cond == "hint")]
print(f"Check against the jiwer library (English): ours {g.werr.sum() / g.wn.sum():.4f}, "
      f"jiwer {jiwer.wer([norm_text(x) for x in g.ref], [norm_text(x) for x in g.hyp]):.4f}")

print("\nWhat language did the baseline think it heard? (rows: true language)")
auto = asr_df[asr_df.cond == "auto"]
display(pd.crosstab(auto.lang, auto.used_lang))

piv = asr_sum.pivot(index="Language", columns="Setting", values="WER %")[list(COND_LABEL.values())]
ax = piv.plot(kind="bar", figsize=(8, 4), rot=0)
ax.set_ylabel("Word error rate % (lower is better)"); ax.set_xlabel(""); ax.set_title(f"Speech to text on FLEURS ({len(SIDS)} sentences per language)")
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, "fig_speech_to_text.png"), dpi=150); plt.show()

### 7b · Does the size of the speech model matter?

`whisper-small` is the model the hackathon brief suggests, and it is fast. Whisper also comes in bigger sizes that have heard far more speech. Our first full run used `whisper-small` and it made too many mistakes in Hindi, Tamil and Bengali, so the main model is now `whisper-large-v3`. Here the same clips go through the second model chosen in the first cell, with the speaker's language given to both, so the only thing that changes is the model.

In [ ]:
#@title 7b · Compare a second speech model
BIG, big_df = None, None
RUN_BIG = bool(COMPARE_ASR_MODEL) and COMPARE_ASR_MODEL != ASR_MODEL and (INFO["device"] == "cuda" or bool(os.environ.get("POLYGLOT_COMPARE_ON_CPU")))
if RUN_BIG:
    t0 = time.time()
    try:
        pc.set_asr_model(COMPARE_ASR_MODEL)
        rows = []
        for l in LANGS_TESTED:
            outs = pc.transcribe_batch([audio[l][s] for s in SIDS], lang_hint=l, with_probs=False, batch_size=8)
            for s, o in zip(SIDS, outs):
                e, n = wer_parts(text[l][s], o["text"])
                ce, cn = cer_parts(text[l][s], o["text"])
                rows.append(dict(lang=l, sid=s, ref=text[l][s], hyp=o["text"], werr=e, wn=n, cerr=ce, cn=cn))
        big_df = pd.DataFrame(rows)
    except Exception as e:
        print("The second model could not be run, so this comparison is skipped:", str(e)[:300])
    finally:
        pc.set_asr_model(ASR_MODEL)            # back to the main model for the rest of the notebook

if big_df is not None:
    small = asr_df[asr_df.cond == "hint"]
    out = []
    for l in LANGS_TESTED:
        a, b = small[small.lang == l], big_df[big_df.lang == l]
        ea, eb, n = a.werr.to_numpy(), b.werr.to_numpy(), a.wn.to_numpy()
        lo, hi = boot_ci(len(a), lambda i: (eb[i].sum() - ea[i].sum()) / max(n[i].sum(), 1))
        out.append({"lang": l, "Language": pc.LANGS[l]["name"],
                    "wer_main": 100 * ea.sum() / n.sum(), "wer_big": 100 * eb.sum() / n.sum(),
                    "change": 100 * (eb.sum() - ea.sum()) / n.sum(), "Change 95% range": f"{100 * lo:+.1f} to {100 * hi:+.1f}",
                    "cer_main": 100 * a.cerr.sum() / a.cn.sum(), "cer_big": 100 * b.cerr.sum() / b.cn.sum()})
    BIG = pd.DataFrame(out)
    m, b = ASR_MODEL.split("/")[-1], COMPARE_ASR_MODEL.split("/")[-1]
    print(f"Done in {time.time() - t0:.0f} s. Both models were given the speaker's language. Lower is better.\n")
    display(BIG.rename(columns={"wer_main": f"WER % {m}", "wer_big": f"WER % {b}", "change": "WER change (points)",
                                "cer_main": f"CER % {m}", "cer_big": f"CER % {b}"}).drop(columns=["lang"]).round(1))
elif not RUN_BIG:
    print("Skipped: no second model chosen, or there is no GPU.")

## 8 · Translation, pair by pair

Here the input is the correct written sentence, so this measures translation alone. Each model is compared with:

- **Baseline: no translation.** Just hand back the source sentence. For two languages in different scripts this scores close to zero, and that is the point: it is the floor every translator must beat.
- **Gemini** (when a key is set), a large general model, as an outside comparison. Gemini may have seen these public sentences on the web, so its score here could be flattering.

In [ ]:
#@title 8 · Translation test
PAIRS = [("hi", "en"), ("ta", "en"), ("bn", "en"), ("en", "hi"), ("en", "ta"), ("en", "bn"), ("hi", "ta")]
ENGINE_LABEL = {"copy": "Baseline: no translation", "nllb": "NLLB-200 (ours)", "gemini": "Gemini (comparison)"}

_gem_path = os.path.join(RESULTS_DIR, "gemini_cache.json")
_gem_cache = json.load(open(_gem_path, encoding="utf8")) if os.path.exists(_gem_path) else {}

def gemini_translate(texts, src, tgt):
    """Gemini translations, remembered on disk so a re-run does not spend the free quota again."""
    key = lambda t: f"{pc.CONFIG['gemini_model']}|{src}|{tgt}|{t}"
    todo = [t for t in dict.fromkeys(texts) if key(t) not in _gem_cache]
    for a in range(0, len(todo), 20):
        part = todo[a:a + 20]
        for t, o in zip(part, pc.translate_batch(part, src, tgt, engine="gemini")):
            _gem_cache[key(t)] = o
        json.dump(_gem_cache, open(_gem_path, "w", encoding="utf8"), ensure_ascii=False)
    return [_gem_cache[key(t)] for t in texts]

GEMINI_OK = bool(GEMINI_API_KEY)
t0 = time.time()
rows = []
for src, tgt in PAIRS:
    srcs, refs = [text[src][s] for s in SIDS], [text[tgt][s] for s in SIDS]
    outs = {"copy": srcs, "nllb": pc.translate_batch(srcs, src, tgt)}
    if GEMINI_OK:
        try:
            outs["gemini"] = gemini_translate(srcs, src, tgt)
        except Exception as e:
            GEMINI_OK = False
            print("Gemini stopped answering, so the rest runs without it:", str(e)[:200])
    for eng, hyps in outs.items():
        for s, i, r, h in zip(SIDS, srcs, refs, hyps):
            rows.append(dict(pair=f"{src}-{tgt}", src=src, tgt=tgt, sid=s, engine=eng, input=i, ref=r, hyp=h, chrf=chrf1(h, r)))
mt_df = pd.DataFrame(rows)

def mt_table(df):
    out = []
    for (pair, eng), g in df.groupby(["pair", "engine"], sort=False):
        score, lo, hi = chrf_ci(g.hyp, g.ref)
        out.append({"pair": pair, "engine": eng, "Pair": f"{pc.LANGS[g.src.iloc[0]]['name']} → {pc.LANGS[g.tgt.iloc[0]]['name']}",
                    "Translator": ENGINE_LABEL[eng], "chrF": score, "lo": lo, "hi": hi,
                    "95% range": f"{lo:.1f} to {hi:.1f}", "Sentences": len(g)})
    return pd.DataFrame(out)

mt_sum = mt_table(mt_df)
print(f"Done in {time.time() - t0:.0f} s. Higher chrF is better.\n")
display(mt_sum.drop(columns=["pair", "engine", "lo", "hi"]).round(1))

piv = mt_sum.pivot(index="Pair", columns="Translator", values="chrF")
piv = piv[[c for c in ENGINE_LABEL.values() if c in piv.columns]].loc[mt_sum.Pair.drop_duplicates()]
ax = piv.plot(kind="bar", figsize=(9, 4), rot=20)
ax.set_ylabel("chrF (higher is better)"); ax.set_xlabel(""); ax.set_title(f"Translation on FLEURS ({len(SIDS)} sentences per pair)")
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, "fig_translation.png"), dpi=150); plt.show()

## 9 · Speech in, translated text out

In a call, translation starts from what Whisper *heard*, mistakes included. This shows how much quality is lost at each step:

- **Perfect transcript → NLLB**: the ceiling, if speech-to-text made no mistakes.
- **Whisper → NLLB**: the real pipeline (a "cascade").
- **Whisper translates directly**: Whisper can also turn speech straight into English, skipping the written Hindi or Tamil.

In [ ]:
#@title 9 · Speech-to-translation test
t0 = time.time()
rows = []
def gold_mt(src, tgt):
    g = mt_df[(mt_df.pair == f"{src}-{tgt}") & (mt_df.engine == "nllb")].set_index("sid")
    return [g.loc[s, "hyp"] for s in SIDS]

for src in [l for l in LANGS_TESTED if l != "en"]:
    refs = [text["en"][s] for s in SIDS]
    heard = [asr_cache[(src, s, src)] for s in SIDS]
    direct = [o["text"] for o in pc.transcribe_batch([audio[src][s] for s in SIDS], lang_hint=src, task="translate", with_probs=False)]
    for name, hyps in {"Perfect transcript → NLLB": gold_mt(src, "en"), "Whisper → NLLB": pc.translate_batch(heard, src, "en"),
                       "Whisper translates directly": direct}.items():
        for s, r, h in zip(SIDS, refs, hyps):
            rows.append(dict(pair=f"{src}-en", src=src, tgt="en", sid=s, system=name, ref=r, hyp=h))
    heard_en = [asr_cache[("en", s, "en")] for s in SIDS]
    refs = [text[src][s] for s in SIDS]
    for name, hyps in {"Perfect transcript → NLLB": gold_mt("en", src), "Whisper → NLLB": pc.translate_batch(heard_en, "en", src)}.items():
        for s, r, h in zip(SIDS, refs, hyps):
            rows.append(dict(pair=f"en-{src}", src="en", tgt=src, sid=s, system=name, ref=r, hyp=h))
s2t_df = pd.DataFrame(rows)

out = []
for (pair, system), g in s2t_df.groupby(["pair", "system"], sort=False):
    score, lo, hi = chrf_ci(g.hyp, g.ref)
    out.append({"pair": pair, "system": system, "Pair": f"{pc.LANGS[g.src.iloc[0]]['name']} speech → {pc.LANGS[g.tgt.iloc[0]]['name']} text",
                "System": system, "chrF": score, "95% range": f"{lo:.1f} to {hi:.1f}"})
s2t_sum = pd.DataFrame(out)
print(f"Done in {time.time() - t0:.0f} s. Higher chrF is better.\n")
S2T_ORDER = ["Perfect transcript → NLLB", "Whisper → NLLB", "Whisper translates directly"]
s2t_table = s2t_sum.pivot(index="Pair", columns="System", values="chrF").loc[s2t_sum.Pair.drop_duplicates(), S2T_ORDER].round(1)
display(s2t_table.astype(object).where(s2t_table.notna(), "n/a"))   # Whisper can only translate into English

In [ ]:
#@title 10 · Speed
n_lat = min(20, len(SIDS))
rows = []
for s in SIDS[:n_lat]:
    clip = audio["hi"][s]
    r = pc.translate_speech(clip, ["en", "ta"], lang_hint="hi")
    rows.append({"clip_s": len(clip) / pc.SR, **r["stageMs"]})
lat_df = pd.DataFrame(rows)
lat_df["total"] = lat_df.asr + lat_df.mt + lat_df.tts
LAT = {k: int(lat_df[k].median()) for k in ["asr", "mt", "tts", "total"]}
print(f"One Hindi speaker, two listeners (English and Tamil), {n_lat} clips of about {lat_df.clip_s.median():.0f} s, on {INFO['gpu'] or 'CPU'}:")
print(f"  speech to text      {LAT['asr']:5d} ms")
print(f"  translation (x2)    {LAT['mt']:5d} ms")
print(f"  voice (x2)          {LAT['tts']:5d} ms")
print(f"  total (median)      {LAT['total']:5d} ms    slowest clip {int(lat_df.total.max())} ms")
print("\nThis is computing time after the speaker stops. It leaves out the network and the wait to notice that they stopped.")

## 11 · Our own data: Hinglish

FLEURS is clean, read-aloud, single-language speech. Real Indian conversation is not. So we made our own test set of Hinglish sentences. The sentences were drafted with Gemini and chosen by us.

**Where the speech comes from.** Clips recorded by team members count as real speech. Clips whose name starts with `tts-` were read by ElevenLabs, a text-to-speech service: they are computer-made, cleaner than a real person, and may not say Hinglish the way a real speaker does. The tables below say how many of each kind were used and score them apart.

**What to put in the data folder** (printed by the setup cell):

- `hinglish/sentences.csv` with the columns `id`, `type`, `text`, `english`. `type` is `hinglish`, `hindi` or `english`. `english` is the correct English translation, checked by us.
- `hinglish/audio/` with one recording per sentence and speaker, named like `007_asha.wav` (sentence id, then your name). Phone recordings (`.m4a`, `.mp3`) work too.

Or tick `RUN_RECORDER` in the first cell, and record straight from the browser with the tool below. `HINGLISH_SHEET` can point to a Google Sheet (shared as "anyone with the link") instead of the csv.

**Fair testing.** We split the *sentences* 40/60. The first part (dev) is only used to choose which method to use. All reported scores come from the second part (test), which played no part in that choice.

In [ ]:
#@title 11 · Load our recordings
def find_recordings(folder=None):
    recs = []
    for p in sorted(glob.glob(os.path.join(folder or HINGLISH_DIR, "audio", "*"))):
        stem, ext = os.path.splitext(os.path.basename(p))
        m = re.match(r"^0*(\d+)(?:[_\- ]+(.+))?$", stem)
        if m and ext.lower() in AUDIO_EXT:
            recs.append(dict(id=int(m.group(1)), speaker=(m.group(2) or "speaker1").lower(), path=p))
    return pd.DataFrame(recs, columns=["id", "speaker", "path"])

sent = load_sentences()
recs = find_recordings()
HAVE_HINGLISH = sent is not None and len(recs) > 0 and len(set(recs.id) & set(sent.index)) >= 5
if not HAVE_HINGLISH:
    print("No Hinglish test set yet, so parts 12 to 14 are skipped.")
    print(f"  sentences: {'none' if sent is None else len(sent)}   recordings: {len(recs)}   folder: {HINGLISH_DIR}")
else:
    rec = recs.merge(sent, left_on="id", right_index=True).reset_index(drop=True)
    h_audio = [pc.load_audio(p) for p in rec.path]
    groups = sorted(set(rec.group))
    random.Random(SEED).shuffle(groups)
    n_dev = max(1, round(0.4 * len(groups)))
    DEV_GROUPS = set(groups[:n_dev])
    rec["split"] = ["dev" if g in DEV_GROUPS else "test" for g in rec.group]
    mins = sum(len(a) for a in h_audio) / pc.SR / 60
    rec["voice"] = ["computer" if s.startswith("tts") else "human" for s in rec.speaker]
    N_HUMAN, N_COMPUTER = int((rec.voice == "human").sum()), int((rec.voice == "computer").sum())
    print(f"{len(rec)} recordings of {rec.id.nunique()} sentences by {rec.speaker.nunique()} speaker(s), {mins:.1f} minutes.")
    print(f"  recorded by people: {N_HUMAN}    computer-made (ElevenLabs text-to-speech): {N_COMPUTER}")
    if N_HUMAN == 0:
        print("  NOTE: every clip here is computer-made speech. These scores show how the system handles a clean")
        print("  synthetic voice reading Hinglish. They are not a test on real speakers, and real speech is usually harder.")
    display(rec.groupby(["type", "split"]).agg(recordings=("id", "size"), sentences=("id", "nunique")))

In [ ]:
#@title 12 · Recording tool (tick RUN_RECORDER in the first cell)
if RUN_RECORDER and sent is None:
    print("Add hinglish/sentences.csv (or HINGLISH_SHEET) first, so the tool knows what to show.")
elif RUN_RECORDER:
    import gradio as gr

    def rec_prompt(i):
        row = sent.iloc[i]
        return f"**Sentence {i + 1} of {len(sent)}** (id {sent.index[i]}, {row['type']})\n\n# {row['text']}"

    def rec_save(i, speaker, clip):
        name = re.sub(r"\W+", "", (speaker or "").lower())
        if not name:
            return i, rec_prompt(i), clip, "Type your first name at the top first."
        if clip is None:
            return i, rec_prompt(i), clip, "Record the sentence first."
        arr = pc.load_audio(clip)
        path = os.path.join(HINGLISH_DIR, "audio", f"{int(sent.index[i]):03d}_{name}.wav")
        with open(path, "wb") as f:
            f.write(pc.wav_bytes(arr))
        j = min(i + 1, len(sent) - 1)
        return j, rec_prompt(j), None, f"Saved {os.path.basename(path)} ({len(arr) / pc.SR:.1f} s)."

    def rec_move(i, step):
        j = max(0, min(len(sent) - 1, i + step))
        return j, rec_prompt(j), None, ""

    with gr.Blocks(title="Polyglot recorder") as rec_app:
        gr.Markdown("## Polyglot recorder\nRead each sentence the way you would say it to a friend. Record only your own voice.")
        idx = gr.State(0)
        speaker = gr.Textbox(label="Your first name")
        prompt = gr.Markdown(rec_prompt(0))
        mic = gr.Audio(sources=["microphone"], type="numpy", label="Record")
        status = gr.Markdown()
        with gr.Row():
            back = gr.Button("Back")
            save = gr.Button("Save and next", variant="primary")
            skip = gr.Button("Skip")
        save.click(rec_save, [idx, speaker, mic], [idx, prompt, mic, status])
        back.click(lambda i: rec_move(i, -1), [idx], [idx, prompt, mic, status])
        skip.click(lambda i: rec_move(i, 1), [idx], [idx, prompt, mic, status])
    rec_app.launch(share=True)
    print("Send the public link only to your teammates. When everyone is done, re-run from part 11.")

## 13 · Does mixing languages break translation?

Each recording goes through four routes to English:

| Route | What happens |
|---|---|
| **Baseline: no translation** | Hand back the words as they were said. Hinglish is already part English, so this is not zero. |
| **Baseline: auto** | Whisper guesses the language, then NLLB translates. The out-of-the-box pipeline. |
| **Hint** | The speaker's language is given (Hindi), then NLLB translates. |
| **Pivot** | Whisper turns the speech straight into English. |
| **Hint + Gemini** | Like Hint, but Gemini does the translation (comparison only). |

We choose between the open routes on the dev sentences, then report every route on the test sentences.

In [ ]:
#@title 13 · Hinglish test
ROUTE_LABEL = {"copy": "Baseline: no translation", "auto": "Baseline: auto → NLLB", "hint": "Hint → NLLB",
               "pivot": "Pivot: Whisper → English", "hint+gemini": "Hint → Gemini (comparison)"}
OPEN_ROUTES = ["auto", "hint", "pivot"]
HING = None

if HAVE_HINGLISH:
    t0 = time.time()
    hp = pc.detect_language_batch(h_audio)
    rec["auto_lang"] = [pick(p) for p in hp]
    rec["mix_score"] = [pc.mix_score(p) for p in hp]
    rec["hint_lang"] = rec["type"].map({"hinglish": "hi", "hindi": "hi", "english": "en"}).fillna("hi")

    def hear(langs):
        out = [""] * len(rec)
        for lang in sorted(set(langs)):
            idx = [i for i, l in enumerate(langs) if l == lang]
            for i, o in zip(idx, pc.transcribe_batch([h_audio[i] for i in idx], lang_hint=lang, with_probs=False)):
                out[i] = o["text"]
        return out

    def to_english(texts, langs, engine="nllb"):
        out = list(texts)
        for lang in sorted(set(langs) - {"en"}):
            idx = [i for i, l in enumerate(langs) if l == lang]
            part = [texts[i] for i in idx]
            done = gemini_translate(part, lang, "en") if engine == "gemini" else pc.translate_batch(part, lang, "en")
            for i, o in zip(idx, done):
                out[i] = o
        return out

    rec["heard_auto"] = hear(list(rec.auto_lang))
    rec["heard_hint"] = hear(list(rec.hint_lang))
    rec["latin_ratio"] = [pc.latin_ratio(t) for t in rec.heard_hint]
    rec["out_copy"] = rec["text"]
    rec["out_auto"] = to_english(list(rec.heard_auto), list(rec.auto_lang))
    rec["out_hint"] = to_english(list(rec.heard_hint), list(rec.hint_lang))
    rec["out_pivot"] = [o["text"] for o in pc.transcribe_batch(h_audio, lang_hint=list(rec.hint_lang), task="translate", with_probs=False)]
    routes = ["copy", "auto", "hint", "pivot"]
    if GEMINI_OK:
        try:
            rec["out_hint+gemini"] = to_english(list(rec.heard_hint), list(rec.hint_lang), engine="gemini")
            routes.append("hint+gemini")
        except Exception as e:
            print("Gemini skipped:", str(e)[:200])

    mixed = rec[rec.type == "hinglish"]
    dev, test = mixed[mixed.split == "dev"], mixed[mixed.split == "test"]
    dev_scores = {r: chrf(dev["out_" + r], dev.english) for r in OPEN_ROUTES}
    OURS = max(dev_scores, key=dev_scores.get)
    print(f"Done in {time.time() - t0:.0f} s.")
    print("Dev sentences (used only to choose):", {ROUTE_LABEL[r]: round(v, 1) for r, v in dev_scores.items()})
    print(f"Chosen route: {ROUTE_LABEL[OURS]}\n")

    out = []
    for r in routes:
        score, lo, hi = chrf_ci(test["out_" + r], test.english)
        gain, glo, ghi = chrf_gain_ci(test.out_auto, test["out_" + r], test.english)
        out.append({"route": r, "Route": ROUTE_LABEL[r] + ("  ← ours" if r == OURS else ""), "chrF": score,
                    "95% range": f"{lo:.1f} to {hi:.1f}", "vs baseline auto": gain, "gain range": f"{glo:+.1f} to {ghi:+.1f}"})
    hing_sum = pd.DataFrame(out)
    print(f"TEST sentences: {test.id.nunique()} sentences, {len(test)} recordings. Higher chrF is better.")
    display(hing_sum.drop(columns=["route"]).round(1))

    print("What language did Whisper think our Hinglish was?")
    display(test.auto_lang.value_counts().rename("recordings").to_frame().T)

    print("By speaker (test):")
    display(test.groupby("speaker").apply(lambda g: pd.Series({
        "recordings": len(g), "baseline auto chrF": chrf(g.out_auto, g.english), "ours chrF": chrf(g["out_" + OURS], g.english)})).round(1))

    print("By kind of voice (test). 'computer' means ElevenLabs text-to-speech, not a person:")
    display(test.groupby("voice").apply(lambda g: pd.Series({
        "recordings": len(g), "baseline auto chrF": chrf(g.out_auto, g.english), "ours chrF": chrf(g["out_" + OURS], g.english)})).round(1))

    HING = {"ours": OURS, "dev": dev_scores, "table": hing_sum, "test": test, "routes": routes}

In [ ]:
#@title 13b · Same speakers, three ways of talking (needs 'hindi' and 'english' control sentences)
CONTROLS = None
if HAVE_HINGLISH and rec[rec.split == "test"].type.nunique() > 1:
    tst = rec[rec.split == "test"]
    out = []
    for typ, g in tst.groupby("type"):
        out.append({"How it was said": typ, "Recordings": len(g),
                    "Baseline auto chrF": chrf(g.out_auto, g.english), "Ours chrF": chrf(g["out_" + OURS], g.english),
                    "Whisper picked the speaker's language %": 100 * float((g.auto_lang == g.hint_lang).mean())})
    CONTROLS = pd.DataFrame(out)
    print("Same speakers and microphones, so the differences come from the way of talking:")
    display(CONTROLS.round(1))

    # Can we tell Hinglish from plain Hindi automatically?  Threshold chosen on dev, scored on test.
    hh = rec[rec.type.isin(["hinglish", "hindi"])]
    if hh[hh.split == "test"].type.nunique() == 2 and hh[hh.split == "dev"].type.nunique() == 2:
        def auc(pos, neg):
            pos, neg = np.asarray(pos, float), np.asarray(neg, float)
            return float(((pos[:, None] > neg[None, :]).sum() + 0.5 * (pos[:, None] == neg[None, :]).sum()) / (len(pos) * len(neg)))
        d, t = hh[hh.split == "dev"], hh[hh.split == "test"]
        det = []
        for feat, label in [("mix_score", "Whisper's language probabilities"), ("latin_ratio", "English-letter words in the transcript")]:
            cands = sorted(set(d[feat]))
            acc = lambda th, df: float(((df[feat] >= th) == (df.type == "hinglish")).mean())
            th = max(cands, key=lambda x: acc(x, d))
            det.append({"Signal": label, "AUC (test)": auc(t[t.type == "hinglish"][feat], t[t.type == "hindi"][feat]),
                        "Threshold (from dev)": th, "Accuracy % (test)": 100 * acc(th, t),
                        "Baseline: always say Hinglish %": 100 * float((t.type == "hinglish").mean())})
        DETECT = pd.DataFrame(det)
        print("\nSpotting Hinglish automatically (AUC 0.5 = guessing, 1.0 = perfect):")
        display(DETECT.round(3))
elif HAVE_HINGLISH:
    print("No 'hindi' or 'english' control sentences were recorded, so this comparison is skipped.")

### 13c · A second test on new sentences

The first test left us with a lead, not a result. On its 18 test sentences, Whisper translating directly scored higher than the baseline, but our own rule had not chosen it. We had already seen those scores, so we could not simply switch methods: that would be choosing the winner after the race.

So we wrote the rules down first (`SECOND_TEST_PLAN.md` in our repository, added before this part was ever run) and then tested on sentences that had never been used.

- **Sentences:** the rest of our draft list, 49 sentences, picked by rule: everything the first test did not use, minus one sentence whose answer key was unclear.
- **Question:** does **Whisper → English directly** beat the **baseline (auto → NLLB)**?
- **Rule:** we call it confirmed only if the whole 95% range of the difference is above zero. The range comes from re-sampling *sentences*, so two voices saying the same sentence are not counted as two independent tests.
- Whatever the result, it is reported below, and the first test stays as it was.

In [ ]:
#@title 13c · Second test on new sentences
CONFIRM = None
try:
    c_sent = load_sentences(CONFIRM_DIR)
    c_recs = find_recordings(CONFIRM_DIR)
    crec = pd.DataFrame()
    if c_sent is not None and len(c_recs):
        crec = (c_recs.merge(c_sent[c_sent.type == "hinglish"], left_on="id", right_index=True)
                .sort_values(["id", "speaker"]).reset_index(drop=True))
    if len(crec) < 10:
        print(f"No second test set yet ({len(crec)} usable recordings in {CONFIRM_DIR}), so this part is skipped.")
    else:
        t0 = time.time()
        c_audio = [pc.load_audio(p) for p in crec.path]
        crec["voice"] = ["computer" if s.startswith("tts") else "human" for s in crec.speaker]
        crec["auto_lang"] = [pick(p) for p in pc.detect_language_batch(c_audio)]

        # Baseline, exactly as in part 13: Whisper guesses the language and writes the words, then NLLB translates.
        heard = [""] * len(crec)
        for lang in sorted(set(crec.auto_lang)):
            idx = [i for i, l in enumerate(crec.auto_lang) if l == lang]
            for i, o in zip(idx, pc.transcribe_batch([c_audio[i] for i in idx], lang_hint=lang, with_probs=False)):
                heard[i] = o["text"]
        out_auto = list(heard)
        for lang in sorted(set(crec.auto_lang) - {"en"}):
            idx = [i for i, l in enumerate(crec.auto_lang) if l == lang]
            for i, o in zip(idx, pc.translate_batch([heard[i] for i in idx], lang, "en")):
                out_auto[i] = o
        crec["heard_auto"], crec["out_copy"], crec["out_auto"] = heard, crec["text"], out_auto
        # Direct, exactly as "pivot" in part 13: Whisper turns the speech straight into English.
        crec["out_pivot"] = [o["text"] for o in pc.transcribe_batch(c_audio, lang_hint=["hi"] * len(crec), task="translate", with_probs=False)]

        A, P, R = list(crec.out_auto), list(crec.out_pivot), list(crec.english)
        def gain_on(rows):
            return chrf([P[j] for j in rows], [R[j] for j in rows]) - chrf([A[j] for j in rows], [R[j] for j in rows])
        ids = crec.id.to_numpy()
        uniq = np.unique(ids)
        rows_of = {s: np.flatnonzero(ids == s) for s in uniq}
        rng = np.random.default_rng(SEED)
        boots = [gain_on(np.concatenate([rows_of[s] for s in rng.choice(uniq, len(uniq))])) for _ in range(500)]
        gain, glo, ghi = float(gain_on(np.arange(len(crec)))), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))
        verdict = ("confirmed: direct translation is better" if glo > 0 else
                   "the baseline is better" if ghi < 0 else "not confirmed")

        a_s, a_lo, a_hi = chrf_ci(A, R)
        p_s, p_lo, p_hi = chrf_ci(P, R)
        c_s = float(chrf(crec.out_copy, R))
        n_h, n_c = int((crec.voice == "human").sum()), int((crec.voice == "computer").sum())
        print(f"Done in {time.time() - t0:.0f} s. {crec.id.nunique()} new sentences, {len(crec)} recordings "
              f"({n_h} by people, {n_c} computer-made). Higher chrF is better.")
        display(pd.DataFrame([
            {"Route": ROUTE_LABEL["copy"], "chrF": c_s, "95% range": ""},
            {"Route": ROUTE_LABEL["auto"], "chrF": a_s, "95% range": f"{a_lo:.1f} to {a_hi:.1f}"},
            {"Route": ROUTE_LABEL["pivot"], "chrF": p_s, "95% range": f"{p_lo:.1f} to {p_hi:.1f}"}]).round(1))
        print(f"Direct minus baseline: {gain:+.1f} chrF points. 95% range, re-sampling sentences: {glo:+.1f} to {ghi:+.1f}.")
        print("RESULT BY OUR RULE:", verdict.upper())
        print("\nBy voice:")
        display(crec.groupby("speaker").apply(lambda g: pd.Series({
            "recordings": len(g), "baseline chrF": chrf(g.out_auto, g.english), "direct chrF": chrf(g.out_pivot, g.english)})).round(1))
        print("What language did Whisper think these clips were?")
        display(crec.auto_lang.value_counts().rename("recordings").to_frame().T)

        summary = (f"Second test on {crec.id.nunique()} new sentences ({len(crec)} clips): Whisper translating directly scored "
                   f"{p_s:.1f} chrF and the baseline {a_s:.1f}, a difference of {gain:+.1f} (95% range {glo:+.1f} to {ghi:+.1f}). "
                   f"By the rule we wrote down beforehand: {verdict}.")
        CONFIRM = {"question": "On new sentences, does Whisper translating directly beat the baseline (auto, then NLLB)?",
                   "n": int(len(crec)), "sentences": int(crec.id.nunique()), "speakers": int(crec.speaker.nunique()),
                   "human_recordings": n_h, "computer_recordings": n_c,
                   "copy": round(c_s, 2), "baseline": round(float(a_s), 2), "direct": round(float(p_s), 2),
                   "gain": round(gain, 2), "gain_range": [round(glo, 2), round(ghi, 2)], "verdict": verdict,
                   "summary": summary, "rec": crec}
except Exception as e:
    CONFIRM = None
    print("The second test could not be finished, so it is left out:", repr(e)[:600])

## 14 · Checked by real speakers

chrF only counts matching letters. It cannot tell whether a translation *means* the right thing. So team members who speak Hindi and English rate the meaning themselves.

- The rater sees what was said, the correct English, and two translations called **A** and **B**. Which system wrote which is hidden and shuffled for every sentence.
- **2** = the meaning is right, **1** = partly right, **0** = wrong or nonsense.
- One clip per sentence is rated. If the second test ran, its sentences are the ones rated, comparing the baseline with direct translation.

The tool below opens a small page for rating. Every answer is saved straight away. After rating, run cell 14b and the Save cell again.

In [ ]:
#@title 14 · Blind rating tool
RATE = None
try:
    if CONFIRM is not None:
        r_src, r_name, r_pair = CONFIRM["rec"], "second test", ("auto", "pivot")
    elif HING is not None:
        r_src, r_name, r_pair = HING["test"], "first test", ("auto", OURS if OURS != "auto" else "pivot")
    else:
        r_src = None
    if r_src is None:
        print("No Hinglish outputs to rate yet.")
    else:
        pool = r_src.sort_values(["id", "speaker"]).reset_index(drop=True)
        keep = [g.index[n % len(g)] for n, (_, g) in enumerate(pool.groupby("id", sort=True))]   # voices take turns
        pool = pool.loc[keep].reset_index(drop=True)
        rng = random.Random(SEED)
        sheet, key = [], {}
        for k, row in pool.iterrows():
            a, b = r_pair if rng.random() < 0.5 else r_pair[::-1]
            key[str(k)] = {"A": a, "B": b}
            sheet.append({"row": k, "said": row["text"], "correct_english": row["english"],
                          "output_A": row["out_" + a], "output_B": row["out_" + b], "rating_A": "", "rating_B": "", "notes": ""})
        RATE = {"sheet": pd.DataFrame(sheet), "key": key, "set": r_name, "pair": r_pair}
        RATE["sheet"].to_csv(os.path.join(RESULTS_DIR, "ratings_sheet.csv"), index=False, encoding="utf-8-sig")
        json.dump(key, open(os.path.join(RESULTS_DIR, "ratings_key.json"), "w"))
        print(f"{len(sheet)} sentences from the {r_name} are ready to rate: {ROUTE_LABEL[r_pair[0]]}  against  {ROUTE_LABEL[r_pair[1]]}.")
except Exception as e:
    RATE = None
    print("The rating sheet could not be made:", repr(e)[:600])

def rate_show(i):
    r = RATE["sheet"].iloc[i]
    head = (f"**Sentence {i + 1} of {len(RATE['sheet'])}**\n\n**What was said:** {r['said']}\n\n"
            f"**What it means:** {r['correct_english']}")
    return head, str(r["output_A"]), str(r["output_B"])

def rate_save(i, rater, ra, rb):
    name = re.sub(r"\W+", "", (rater or "").lower())
    if not name:
        return (i, *rate_show(i), ra, rb, "Type your first name at the top first.")
    if ra is None or rb is None:
        return (i, *rate_show(i), ra, rb, "Give both A and B a score first.")
    path = os.path.join(RESULTS_DIR, f"ratings_filled_{name}.csv")
    df = pd.read_csv(path) if os.path.exists(path) else RATE["sheet"].copy()
    if len(df) != len(RATE["sheet"]) or list(df["said"]) != list(RATE["sheet"]["said"]):
        df = RATE["sheet"].copy()                 # an older sheet with other sentences: start this rater again
    for col, val in (("rating_A", ra), ("rating_B", rb)):
        df[col] = df[col].astype(object)
        df.loc[df["row"] == i, col] = int(val)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    done = int(pd.to_numeric(df["rating_A"], errors="coerce").notna().sum())
    j = min(i + 1, len(df) - 1)
    note = f"Saved. You have rated {done} of {len(df)}." + ("  All done, thank you!" if done == len(df) else "")
    return (j, *rate_show(j), None, None, note)

def rate_move(i, step):
    j = max(0, min(len(RATE["sheet"]) - 1, i + step))
    return (j, *rate_show(j), None, None, "")

RATE_APP_OK = False
if RATE is not None and RUN_RATER:
    try:
        import gradio as gr
        with gr.Blocks(title="Polyglot ratings") as rate_app:
            gr.Markdown("## Polyglot: rate the translations\nYou see what was said, what it means, and two translations called A and B. "
                        "You are not told which system wrote which. Score the **meaning** of each one: "
                        "**2** = right, **1** = partly right, **0** = wrong or nonsense.")
            r_idx = gr.State(0)
            r_who = gr.Textbox(label="Your first name")
            _h, _a, _b = rate_show(0)
            r_head = gr.Markdown(_h)
            with gr.Row():
                with gr.Column():
                    r_a = gr.Textbox(label="Translation A", value=_a, interactive=False, lines=3)
                    r_sa = gr.Radio([0, 1, 2], label="Score for A")
                with gr.Column():
                    r_b = gr.Textbox(label="Translation B", value=_b, interactive=False, lines=3)
                    r_sb = gr.Radio([0, 1, 2], label="Score for B")
            r_note = gr.Markdown()
            with gr.Row():
                r_back = gr.Button("Back")
                r_next = gr.Button("Save and next", variant="primary")
            _outs = [r_idx, r_head, r_a, r_b, r_sa, r_sb, r_note]
            r_next.click(rate_save, [r_idx, r_who, r_sa, r_sb], _outs)
            r_back.click(lambda i: rate_move(i, -1), [r_idx], _outs)
        rate_app.launch(share=True)
        RATE_APP_OK = True
        print("Open the public link (send it to the teammates who will rate). Each person types their own first name.")
        print("When everyone has finished, run cell 14b and then the Save cell again.")
    except Exception as e:
        print("The rating page could not be opened:", repr(e)[:600])
        print("You can still rate by hand: open ratings_sheet.csv in", RESULTS_DIR)
        print("Fill in rating_A and rating_B with 0, 1 or 2, and save it there as ratings_filled_yourname.csv.")

In [ ]:
#@title 14b · Count the ratings (run this again after people have rated)
RATINGS, RATING_INFO = None, None
try:
    got, skipped = [], 0
    if RATE is not None:
        sh = RATE["sheet"].set_index("row")
        for f in sorted(glob.glob(os.path.join(RESULTS_DIR, "ratings_filled*.csv"))):
            who = os.path.splitext(os.path.basename(f))[0].replace("ratings_filled", "").strip("_") or "rater"
            for _, r in pd.read_csv(f).iterrows():
                va, vb = pd.to_numeric(r.get("rating_A"), errors="coerce"), pd.to_numeric(r.get("rating_B"), errors="coerce")
                if pd.isna(va) or pd.isna(vb):
                    continue
                k = int(r["row"])
                if k not in sh.index or str(sh.loc[k, "said"]) != str(r["said"]) or str(sh.loc[k, "output_A"]) != str(r["output_A"]):
                    skipped += 1                      # rated on an older sheet: do not mix it in
                    continue
                sides = RATE["key"][str(k)]
                got.append({"rater": who, "row": k, sides["A"]: float(va), sides["B"]: float(vb)})
    got = pd.DataFrame(got)
    if skipped:
        print(f"{skipped} ratings were left out because they belong to an older rating sheet.")
    if len(got):
        base, new = RATE["pair"]
        per = got.groupby("row")[[base, new]].mean()          # average the raters for each sentence
        RATINGS = pd.DataFrame({"mean_rating": [got[base].mean(), got[new].mean()],
                                "fully_right": [100 * float((got[base] == 2).mean()), 100 * float((got[new] == 2).mean())],
                                "rated": [int(len(got)), int(len(got))]}, index=[ROUTE_LABEL[base], ROUTE_LABEL[new]])
        RATINGS.index.name = "system"
        diff = (per[new] - per[base]).to_numpy()
        lo, hi = boot_ci(len(diff), lambda i: float(diff[i].mean()), 1000)
        pts = [(float(per.loc[k, RATE["key"][str(k)][s]]), chrf1(sh.loc[k, f"output_{s}"], sh.loc[k, "correct_english"]))
               for k in per.index for s in "AB"]
        rho = float(pd.Series([p[0] for p in pts]).corr(pd.Series([p[1] for p in pts]), method="spearman"))
        RATING_INFO = {"rated_set": RATE["set"], "raters": int(got.rater.nunique()), "sentences": int(len(per)),
                       "compared": [ROUTE_LABEL[base], ROUTE_LABEL[new]],
                       "second_better": int((diff > 0).sum()), "same": int((diff == 0).sum()), "first_better": int((diff < 0).sum()),
                       "mean_difference": round(float(diff.mean()), 3), "difference_range": [round(lo, 3), round(hi, 3)],
                       "agreement_with_chrf": None if rho != rho else round(rho, 2)}
        print(f"Blind ratings by {RATING_INFO['raters']} team member(s) on {len(per)} sentences of the {RATE['set']} (0 to 2):")
        display(RATINGS.round(2))
        print(f"Sentence by sentence: {ROUTE_LABEL[new]} was rated higher on {RATING_INFO['second_better']}, the same on "
              f"{RATING_INFO['same']}, lower on {RATING_INFO['first_better']}.")
        print(f"Average difference: {diff.mean():+.2f} points (95% range {lo:+.2f} to {hi:+.2f}).")
        print("Do the ratings agree with chrF? Spearman correlation:", RATING_INFO["agreement_with_chrf"])
    else:
        print("No ratings yet. Use the rating page from cell 14, then run this cell and the Save cell again.")
except Exception as e:
    RATINGS, RATING_INFO = None, None
    print("The ratings could not be counted:", repr(e)[:600])

## 15 · Where it fails

The five worst results for each language, with the model's output next to the correct answer. The "why" column is an automatic first guess. We read each one ourselves and wrote what we found in the write-up.

In [ ]:
#@title 15 · The worst mistakes, side by side
def why_guess(inp, hyp, ref, used=None, true=None):
    notes = []
    if not str(hyp).strip():
        notes.append("no output at all")
    if used and true and used != true:
        notes.append(f"heard the wrong language ({used})")
    if re.search(r"\d", f"{inp} {ref}"):
        notes.append("has numbers")
    if len(str(hyp)) < 0.6 * len(str(ref)):
        notes.append("part of the sentence is missing")
    if len(str(hyp)) > 1.6 * len(str(ref)):
        notes.append("extra or made-up words")
    w = norm_text(hyp).split()
    if len(w) >= 6 and len(set(w)) < 0.5 * len(w):
        notes.append("repeats itself")
    return "auto-guess: " + ("; ".join(notes) if notes else "no obvious pattern, needs a human look")

WORST = []
print("SPEECH TO TEXT (with the speaker's language given)")
for l in LANGS_TESTED:
    g = asr_df[(asr_df.lang == l) & (asr_df.cond == "hint")].copy()
    g["wer"] = g.werr / g.wn.clip(lower=1)
    top = g.sort_values("wer", ascending=False).head(5)
    for _, r in top.iterrows():
        WORST.append({"task": "speech-to-text", "lang": l, "input": f"speech clip {r.sid}", "model": r.hyp, "correct": r.ref,
                      "score": f"WER {100 * r.wer:.0f}%", "why": why_guess("", r.hyp, r.ref)})
    print(f"\n{pc.LANGS[l]['name']}")
    display(top[["wer", "hyp", "ref"]].rename(columns={"wer": "WER", "hyp": "model wrote", "ref": "correct"}).round(2))

print("\nTRANSLATION (NLLB)")
for src, tgt in PAIRS:
    g = mt_df[(mt_df.pair == f"{src}-{tgt}") & (mt_df.engine == "nllb")]
    top = g.sort_values("chrf").head(5)
    for _, r in top.iterrows():
        WORST.append({"task": "translation", "lang": tgt, "pair": f"{src}-{tgt}", "input": r.input, "model": r.hyp, "correct": r.ref,
                      "score": f"chrF {r.chrf:.0f}", "why": why_guess(r.input, r.hyp, r.ref)})
    print(f"\n{pc.LANGS[src]['name']} → {pc.LANGS[tgt]['name']}")
    display(top[["chrf", "input", "hyp", "ref"]].rename(columns={"chrf": "chrF", "hyp": "model wrote", "ref": "correct"}).round(1))

if HING:
    t = HING["test"].copy()
    t["chrf"] = [chrf1(h, r) for h, r in zip(t["out_" + OURS], t.english)]
    top = t.sort_values("chrf").head(5)
    for _, r in top.iterrows():
        WORST.append({"task": "hinglish", "lang": "hinglish", "input": r["text"], "model": r["out_" + OURS], "correct": r["english"],
                      "score": f"chrF {r['chrf']:.0f}", "heard": r["heard_hint"],
                      "why": why_guess(r["text"], r["out_" + OURS], r["english"], r["auto_lang"], r["hint_lang"])})
    print(f"\nHINGLISH → English ({ROUTE_LABEL[OURS]})")
    display(top[["chrf", "text", "heard_hint", "out_" + OURS, "english"]].rename(columns={
        "chrf": "chrF", "text": "said", "heard_hint": "Whisper heard", "out_" + OURS: "model wrote", "english": "correct"}).round(1))

# Order for the results file: our Hinglish failures first, then the worst of each group in turn.
_groups = {}
for w in WORST:
    _groups.setdefault((w["task"], w.get("pair", w["lang"])), []).append(w)
_hing = _groups.pop(("hinglish", "hinglish"), [])
_rest = [g[i] for i in range(5) for g in _groups.values() if i < len(g)]
WORST = _hing + _rest

In [ ]:
#@title 16 · Save the results
def _asr_entry(l):
    g = asr_sum[asr_sum.lang == l].set_index("cond")
    return {"wer": round(g.loc["hint", "WER %"] / 100, 4), "cer": round(g.loc["hint", "CER %"] / 100, 4),
            "wer_auto": round(g.loc["auto", "WER %"] / 100, 4), "wer_room": round(g.loc["room", "WER %"] / 100, 4),
            "lang_right_auto": round(g.loc["auto", "Right language %"] / 100, 4),
            "lang_right_room": round(g.loc["room", "Right language %"] / 100, 4), "n": int(g.loc["hint", "Clips"])}

RESULTS = {
    "generated": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "models": INFO,
    "sentences_per_language": len(SIDS),
    "quick_test": bool(QUICK_TEST or len(SIDS) < 50),
    "asr": {l: _asr_entry(l) for l in LANGS_TESTED},
    "mt": {},
    "s2t": {},
    "latencyMs": {**LAT, "note": "median computing time: one Hindi speaker, English and Tamil listeners"},
    "worst": WORST,
}
for pair, g in mt_sum.groupby("pair", sort=False):
    RESULTS["mt"][pair] = {r.engine: {"chrf": round(r.chrF, 2), "ci": [round(r.lo, 2), round(r.hi, 2)]} for r in g.itertuples()}
    RESULTS["mt"][pair]["n"] = int(g.Sentences.iloc[0])
for r in s2t_sum.itertuples():
    RESULTS["s2t"].setdefault(r.pair, {})[r.system] = round(r.chrF, 2)
if BIG is not None:
    RESULTS["asr_compare_model"] = COMPARE_ASR_MODEL
    for r in BIG.itertuples():
        RESULTS["asr"][r.lang]["wer_compare"] = round(r.wer_big / 100, 4)
        RESULTS["asr"][r.lang]["cer_compare"] = round(r.cer_big / 100, 4)
    big_df.to_csv(os.path.join(RESULTS_DIR, "speech_to_text_outputs_second_model.csv"), index=False, encoding="utf-8-sig")

if HING:
    tab = HING["table"].set_index("route")
    test = HING["test"]
    RESULTS["asr"]["hinglish"] = {"wer": None, "n": int(len(test)),
                                  "note": "No WER: we wrote Hinglish in Latin letters and Whisper writes Devanagari, so word matching would be unfair."}
    RESULTS["hinglish"] = {
        "baseline": {"wer": None, "chrf": round(tab.loc["auto", "chrF"], 2), "name": ROUTE_LABEL["auto"]},
        "fix": {"wer": None, "chrf": round(tab.loc[OURS, "chrF"], 2), "name": ROUTE_LABEL[OURS], "route": OURS,
                "gain": round(tab.loc[OURS, "vs baseline auto"], 2), "gain_range": tab.loc[OURS, "gain range"]},
        "copy": {"chrf": round(tab.loc["copy", "chrF"], 2), "name": ROUTE_LABEL["copy"]},
        "variants": {r: round(tab.loc[r, "chrF"], 2) for r in HING["routes"]},
        "dev": {r: round(v, 2) for r, v in HING["dev"].items()},
        "n": int(len(test)), "sentences": int(test.id.nunique()), "speakers": int(test.speaker.nunique()),
    }
    RESULTS["hinglish"]["speech"] = {"human_recordings": int((test.voice == "human").sum()),
                                     "computer_recordings": int((test.voice == "computer").sum())}
    if RESULTS["hinglish"]["speech"]["computer_recordings"]:
        RESULTS["hinglish"]["speech"]["computer_voice"] = f"ElevenLabs text-to-speech ({ELEVEN_MODEL})"
        RESULTS["hinglish"]["note"] = ("Some or all Hinglish clips are computer-made speech (ElevenLabs), not real speakers. "
                                       "Real speech is usually harder, so read these scores as an upper estimate.")
    if CONTROLS is not None:
        RESULTS["hinglish"]["by_style"] = json.loads(CONTROLS.round(2).to_json(orient="records"))
    notes = [RESULTS["hinglish"].get("note", "")]
    if CONFIRM is not None:
        RESULTS["hinglish"]["confirm"] = {k: v for k, v in CONFIRM.items() if k != "rec"}
        notes.append(CONFIRM["summary"])
        CONFIRM["rec"].drop(columns=["path"]).to_csv(os.path.join(RESULTS_DIR, "hinglish_second_test_outputs.csv"), index=False, encoding="utf-8-sig")
    if RATINGS is not None:
        RESULTS["hinglish"]["human_ratings"] = json.loads(RATINGS.round(3).reset_index().to_json(orient="records"))
        RESULTS["hinglish"]["human_ratings_info"] = RATING_INFO
        m = RATINGS["mean_rating"]
        notes.append(f"Blind ratings by {RATING_INFO['raters']} team member(s) on {RATING_INFO['sentences']} sentences "
                     f"(0 to 2): {m.index[1]} {m.iloc[1]:.2f}, {m.index[0]} {m.iloc[0]:.2f}.")
    if any(notes):
        RESULTS["hinglish"]["note"] = " ".join(n for n in notes if n)
    rec.drop(columns=["path"]).to_csv(os.path.join(RESULTS_DIR, "hinglish_outputs.csv"), index=False, encoding="utf-8-sig")

asr_df.to_csv(os.path.join(RESULTS_DIR, "speech_to_text_outputs.csv"), index=False, encoding="utf-8-sig")
mt_df.to_csv(os.path.join(RESULTS_DIR, "translation_outputs.csv"), index=False, encoding="utf-8-sig")
for path in (os.path.join(RESULTS_DIR, "results.json"), "results.json"):
    with open(path, "w", encoding="utf8") as f:
        json.dump(RESULTS, f, ensure_ascii=False, indent=1)
print("Saved results.json and every model output to", RESULTS_DIR)
print(json.dumps({k: RESULTS.get(k) for k in ("asr", "mt", "hinglish", "latencyMs")}, ensure_ascii=False, indent=1)[:3000])

In [ ]:
#@title 17 · Live demo: speak, then hear the translation
if RUN_DEMO:
    import gradio as gr
    NAMES = {c: v["name"] for c, v in pc.LANGS.items()}

    def demo_run(clip, spoken, target, engine):
        if clip is None:
            return "", "", None, "Record or upload something first."
        r = pc.translate_speech(clip, [target], lang_hint=None if spoken == "auto" else spoken, engine=engine,
                                candidates=LANGS_TESTED)
        out = r["results"][target]
        heard = f"{r['original']}\n[{NAMES.get(r['sourceLang'], r['sourceLang'])}" + (f", {r['mixed']}" if r["mixed"] else "") + "]"
        sound = (pc.SR, pc.load_audio(out["wav"])) if out["wav"] else None
        ms = r["stageMs"]
        return heard, out["text"], sound, f"speech to text {ms['asr']} ms · translation {ms['mt']} ms · voice {ms['tts']} ms"

    with gr.Blocks(title="Polyglot") as demo:
        gr.Markdown("# Polyglot\nSpeak your language. Hear theirs. *A prototype: it makes mistakes, see the results above.*")
        with gr.Row():
            with gr.Column():
                d_mic = gr.Audio(sources=["microphone", "upload"], type="numpy", label="Speak, or upload a clip")
                d_spoken = gr.Dropdown([("Detect it for me", "auto")] + [(NAMES[c], c) for c in LANGS_TESTED], value="hi", label="I am speaking")
                d_target = gr.Dropdown([(NAMES[c], c) for c in LANGS_TESTED], value="en", label="Translate into")
                d_engine = gr.Radio(["nllb"] + (["gemini"] if GEMINI_API_KEY else []), value="nllb", label="Translator")
                d_go = gr.Button("Translate", variant="primary")
            with gr.Column():
                d_heard = gr.Textbox(label="What Polyglot heard", lines=3)
                d_text = gr.Textbox(label="Translation", lines=3)
                d_audio = gr.Audio(label="Translated speech", autoplay=True)
                d_time = gr.Markdown()
        d_go.click(demo_run, [d_mic, d_spoken, d_target, d_engine], [d_heard, d_text, d_audio, d_time])
    demo.launch(share=True)

## What this does not show

- **Small test sets.** 100 sentences per language, and a few dozen Hinglish sentences from a handful of speakers. The 95% ranges show how uncertain each number is.
- **Read speech.** FLEURS and our recordings are read aloud, one clean sentence at a time. Real calls have noise, interruptions and people talking over each other.
- **Not live streaming.** Each sentence is translated after the speaker finishes it. Clips longer than 30 seconds are cut.
- **Numbers and names.** The voice model cannot read digits aloud, and names are often changed.
- **Hinglish has no word error rate here.** We wrote Hinglish in Latin letters, Whisper writes Devanagari, so we score the final English and our own ratings.
- **Gemini may have seen FLEURS.** Those sentences are public, so its scores on them may be too high. Our own recordings do not have this problem.

## Sources

| What | Link | Licence |
|---|---|---|
| Whisper (speech to text) | https://huggingface.co/openai/whisper-large-v3 (compared with https://huggingface.co/openai/whisper-small) | MIT |
| NLLB-200 (translation) | https://huggingface.co/facebook/nllb-200-distilled-600M | CC-BY-NC 4.0 |
| MMS-TTS (voice) | https://huggingface.co/facebook/mms-tts | CC-BY-NC 4.0 |
| FLEURS (test speech and translations) | https://huggingface.co/datasets/google/fleurs | CC-BY 4.0 |
| Gemini API (comparison translator, and first draft of the Hinglish sentences) | https://ai.google.dev | Google terms |
| ElevenLabs (computer-made voices for the Hinglish test clips named `tts-`) | https://elevenlabs.io | ElevenLabs terms |
| Hugging Face Transformers | https://github.com/huggingface/transformers | Apache-2.0 |
| sacrebleu (chrF), jiwer and rapidfuzz (WER) | https://github.com/mjpost/sacrebleu · https://github.com/jitsi/jiwer · https://github.com/rapidfuzz/RapidFuzz | Apache-2.0 · Apache-2.0 · MIT |
| Gradio (demo) | https://gradio.app | Apache-2.0 |

AI tools used while building: Claude (Anthropic), ChatGPT / Codex (OpenAI), Gemini / Antigravity (Google). The Hinglish sentences were drafted with Gemini, then chosen, corrected and recorded by us.